# Alignement explicite métriques / code

Expériences complémentaires : attention croisée bidirectionnelle et alignement hiérarchique entre les métriques et les représentations CodeBERT, Borderline-SMOTE.

> **Exécution** : notebook conçu pour Google Colab. Les chemins pointent vers Google Drive (`/content/drive/MyDrive/bug/...`). Voir [`data/README.md`](../../data/README.md) pour obtenir les données et adapter les chemins.

*Nom d'origine dans Colab : `Untitled89`*


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# Colab: installer dépendances (exécuter une seule fois)
!pip install -q transformers==4.42.0 datasets sentencepiece tokenizers git+https://github.com/huggingface/transformers@main


In [ ]:
!pip install -q torch torchvision torchaudio --upgrade
!pip install -q scikit-learn imbalanced-learn matplotlib seaborn pandas joblib

In [ ]:


# Imports
import os, glob, json, time, random
from datetime import datetime
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope
from imblearn.over_sampling import BorderlineSMOTE
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import gmean
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")


In [ ]:

# Chemins (modifie si besoin)
DRIVE_BASE = '/content/drive/MyDrive/promise_results'  # dossier où seront sauvegardés résultats et modèles
PROMISE_METRICS_DIR = '/content/drive/MyDrive/bug/bug-data'  # dossier métriques CSV
PROMISE_SOURCE_DIR  = '/content/drive/MyDrive/bug/bug-data/code source' # dossier source-code

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, "models"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, "results"), exist_ok=True)

# Projets cibles (intraprojet on fait un projet à la fois)
PROJECTS = ["ant", "camel", "jedit", "log4j", "synapse", "lucene", "poi", "velocity", "xalan", "xerces"]

print("Drive monté. Répertoires :")
print("RESULTS ->", DRIVE_BASE)
print("METRICS ->", PROMISE_METRICS_DIR)
print("SOURCE  ->", PROMISE_SOURCE_DIR)


In [ ]:
# Fonction pour trouver fichiers métriques d'un projet (toutes versions)
def find_project_metric_files(project_name, metrics_dir):
    all_csv = sorted(glob.glob(os.path.join(metrics_dir, "**", "*.csv"), recursive=True))
    project_files = [f for f in all_csv if os.path.basename(f).lower().startswith(project_name.lower()) or (project_name.lower() in os.path.basename(f).lower())]
    return project_files

# Fonctions pour trouver le fichier source correspondant à une ligne de métriques
# On suppose que le CSV contient une colonne 'file' ou 'name' ou 'filename' désignant le fichier Java.
def possible_name_columns(df):
    for c in ['file', 'name', 'filename', 'path', 'class']:
        if c in df.columns:
            return c
    return None

# Construire un index des fichiers sources (basename -> full path) pour accélérer correspondance
def build_source_index(source_root):
    idx = {}
    exts = ('.java', '.py', '.c', '.cpp')  # PROMISE = Java normalement
    for root, _, files in os.walk(source_root):
        for f in files:
            if f.lower().endswith(exts):
                key = f.lower()
                if key not in idx:
                    idx[key] = []
                idx[key].append(os.path.join(root, f))
    return idx

print("Indexation source (peut prendre du temps) - si grand ensemble, ce sera lent...")
source_index = build_source_index(PROMISE_SOURCE_DIR)
print("Index construit: %d basenames trouvés" % len(source_index))


In [ ]:
def load_and_concat_metric_csvs(files):
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            df['_source_file_csv'] = os.path.basename(f)
            dfs.append(df)
        except Exception as e:
            print(f"Erreur lecture {f}: {e}")
    if not dfs:
        return pd.DataFrame()
    df_all = pd.concat(dfs, ignore_index=True)
    return df_all

def split_intraproject_by_version(df):
    # On s'attend à une colonne 'version' ou un nom de fichier contenant la version
    if 'version' in df.columns:
        last_version = sorted(df['version'].unique())[-1]
        train_df = df[df['version'] != last_version].copy()
        val_df = df[df['version'] == last_version].copy()
        return train_df, val_df
    else:
        # Fallback: si les CSV contiennent indique de version dans le nom du csv, on utilise _source_file_csv
        versions = df['_source_file_csv'].unique()
        versions_sorted = sorted(versions)
        if len(versions_sorted) == 1:
            # Un seul CSV -> split temporel aléatoire stratifié (20% val)
            from sklearn.model_selection import train_test_split
            train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['bug'] if 'bug' in df.columns else None)
            return train_df, val_df
        last_csv = versions_sorted[-1]
        train_df = df[df['_source_file_csv'] != last_csv].copy()
        val_df = df[df['_source_file_csv'] == last_csv].copy()
        return train_df, val_df

def prepare_features_and_labels(df, feature_cols=None, target_col='bug'):
    if feature_cols is None:
        feature_cols = [c for c in df.columns if c != target_col and not c.startswith('_') and c not in ('name','file','filename','path','version')]
    # Fill missing
    df[feature_cols] = df[feature_cols].fillna(df[feature_cols].mean())
    X = df[feature_cols].values
    y = df[target_col].values.astype(int)
    return X, y, feature_cols

def remove_outliers_elliptic(X, y, contamination=0.05):
    # Apply on majority class only
    df = pd.DataFrame(X)
    df['target'] = y
    counts = df['target'].value_counts()
    if len(counts) != 2:
        # fallback: return original
        return X, y
    maj_label = counts.idxmax()
    maj_df = df[df['target'] == maj_label].drop('target', axis=1)
    if len(maj_df) < 10:
        return X, y
    ee = EllipticEnvelope(contamination=contamination, random_state=42)
    try:
        preds = ee.fit_predict(maj_df)
        maj_clean = maj_df[preds == 1]
        min_df = df[df['target'] != maj_label].drop('target', axis=1)
        # balance by undersampling majority to minority size if needed
        if len(maj_clean) > len(min_df):
            maj_clean = maj_clean.sample(n=len(min_df), random_state=42)
        else:
            min_df = min_df.sample(n=len(maj_clean), random_state=42)
        balanced = pd.concat([maj_clean, min_df], ignore_index=True)
        labels = [maj_label]*len(maj_clean) + [1-maj_label]*len(min_df)
        Xr = balanced.values
        yr = np.array(labels)
        # shuffle
        idx = np.random.RandomState(42).permutation(len(yr))
        return Xr[idx], yr[idx]
    except Exception as e:
        print("EllipticEnvelope error:", e)
        return X, y

def standardize_train_val(X_train, X_val):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    return X_train_s, X_val_s, scaler


In [ ]:
# Tokenizer + model pour le code (CodeBERT)
CODE_MODEL_NAME = "microsoft/codebert-base"
tokenizer_code = AutoTokenizer.from_pretrained(CODE_MODEL_NAME)
model_code = AutoModel.from_pretrained(CODE_MODEL_NAME).eval().to('cuda' if torch.cuda.is_available() else 'cpu')

# Fonction pour obtenir embedding moyen des tokens d'un fichier source
def embed_code_file(path, tokenizer, model, max_length=256, device='cpu'):
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            txt = f.read()
    except:
        return None
    # Truncate long files
    if len(txt) > 20000:
        txt = txt[:20000]
    toks = tokenizer(txt, truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    toks = {k: v.to(device) for k,v in toks.items()}
    with torch.no_grad():
        out = model(**toks, output_hidden_states=False, return_dict=True)
        emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()  # pooled mean
    return emb

# Build mapping from metric row -> code embedding (attempt match via filename columns)
def map_metrics_to_code_embeddings(df, source_index, tokenizer, model, device='cpu'):
    name_col = possible_name_columns(df)
    embeddings = []
    matched_idx = []
    for idx, row in df.iterrows():
        found_path = None
        if name_col:
            candidate = str(row.get(name_col, "")).strip()
            # try exact basename match
            base = os.path.basename(candidate).lower()
            if base in source_index:
                found_path = source_index[base][0]  # take first
            else:
                # try contains search in index keys
                for k, paths in source_index.items():
                    if base and base in k:
                        found_path = paths[0]
                        break
        # fallback: try matching by class name in row or by file column content
        if not found_path:
            # try find any path containing a token from candidate (slow)
            candidate_tokens = [t for t in candidate.replace('/', ' ').split() if len(t) > 3]
            for tok in candidate_tokens:
                for k, paths in source_index.items():
                    if tok.lower() in k:
                        found_path = paths[0]
                        break
                if found_path:
                    break
        if not found_path:
            embeddings.append(None)
            continue
        emb = embed_code_file(found_path, tokenizer, model, device=device)
        embeddings.append(emb)
        matched_idx.append(idx)
    return embeddings, matched_idx


In [ ]:
class MetricsCodeDataset(Dataset):
    def __init__(self, X_metrics, X_code_embeddings, y):
        # X_code_embeddings may contain None entries -> we remove those rows earlier
        self.Xm = torch.tensor(X_metrics, dtype=torch.float32)
        self.Xc = torch.tensor(X_code_embeddings, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.Xm[idx], self.Xc[idx], self.y[idx]


In [ ]:
class FusionModel(nn.Module):
    def __init__(self, metrics_dim, code_emb_dim, config):
        super(FusionModel, self).__init__()
        self.config = config
        self.metrics_proj = nn.Sequential(
            nn.Linear(metrics_dim, config.EMBEDDING_DIM, bias=True),
            nn.ReLU(),
            nn.LayerNorm(config.EMBEDDING_DIM)
        )
        # small transformer encoder for metrics (seq_len=1 -> treated as token)
        encoder_layer = nn.TransformerEncoderLayer(d_model=config.EMBEDDING_DIM,
                                                   nhead=max(1, config.N_HEADS),
                                                   dropout=config.DROPOUT,
                                                   batch_first=True)
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=max(1, config.NUM_LAYERS),
                                                     norm=nn.LayerNorm(config.EMBEDDING_DIM))
        # project code embedding to same dim
        self.code_proj = nn.Linear(code_emb_dim, config.EMBEDDING_DIM)
        # cross-attention: we implement MultiheadAttention where query=code, key/value=metrics
        self.cross_attn = nn.MultiheadAttention(embed_dim=config.EMBEDDING_DIM,
                                                num_heads=max(1, config.N_HEADS),
                                                batch_first=True)
        # classifier after fusion
        self.classifier = nn.Sequential(
            nn.Linear(config.EMBEDDING_DIM, 4*config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(4*config.EMBEDDING_DIM, config.NUM_CLASSES)
        )
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, xm, xc):
        # xm: (batch, metrics_dim)
        # xc: (batch, code_emb_dim)
        m = self.metrics_proj(xm)             # (batch, embed)
        # make seq_len=1 for metrics encoder
        m = m.unsqueeze(1)                   # (batch, 1, embed)
        m_enc = self.metrics_encoder(m)      # (batch, 1, embed)
        c = self.code_proj(xc).unsqueeze(1)  # (batch, 1, embed)
        # Cross-attention: queries = c, keys/values = m_enc
        # MultiheadAttention expects (batch, seq, embed) with batch_first=True
        attn_out, attn_w = self.cross_attn(query=c, key=m_enc, value=m_enc, need_weights=True)
        # attn_out: (batch, 1, embed) -> squeeze
        fused = attn_out.squeeze(1)
        logits = self.classifier(fused)
        return self.logsoftmax(logits)


In [ ]:
def calculate_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2,2):
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) else 0
        specificity = tn / (tn + fp) if (tn + fp) else 0
        precision = tp / (tp + fp) if (tp + fp) else 0
        f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) else 0
        g = gmean([max(sensitivity,1e-12), max(specificity,1e-12)])
        return {'cm': cm, 'sensitivity': sensitivity, 'specificity': specificity, 'precision': precision, 'f1': f1, 'gmean': g}
    else:
        return {'cm': cm}

def train_model(model, train_loader, val_loader, config, device, save_prefix):
    criterion = nn.NLLLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE)
    history = {'train_loss':[], 'val_loss':[], 'train_gmean':[], 'val_gmean':[]}
    best_g = -1.0
    best_state = None
    patience = 0

    for epoch in range(config.EPOCHS):
        # training
        model.train()
        train_loss = 0.0
        all_preds, all_labels = [], []
        for xm, xc, y in train_loader:
            xm = xm.to(device); xc = xc.to(device); y = y.to(device)
            optimizer.zero_grad()
            out = model(xm, xc)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xm.size(0)
            preds = out.argmax(dim=1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.detach().cpu().numpy())
        train_loss = train_loss / len(train_loader.dataset)
        tr_metrics = calculate_metrics(all_labels, all_preds)

        # validation
        model.eval()
        val_loss = 0.0
        v_preds, v_labels = [], []
        with torch.no_grad():
            for xm, xc, y in val_loader:
                xm = xm.to(device); xc = xc.to(device); y = y.to(device)
                out = model(xm, xc)
                loss = criterion(out, y)
                val_loss += loss.item() * xm.size(0)
                preds = out.argmax(dim=1).cpu().numpy()
                v_preds.extend(preds)
                v_labels.extend(y.cpu().numpy())
        val_loss = val_loss / len(val_loader.dataset)
        val_metrics = calculate_metrics(v_labels, v_preds)

        history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
        history['train_gmean'].append(tr_metrics.get('gmean', 0)); history['val_gmean'].append(val_metrics.get('gmean', 0))

        if (epoch+1) % 10 == 0 or epoch==0:
            print(f"Epoch {epoch+1}/{config.EPOCHS} - train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_gmean={val_metrics.get('gmean',0):.4f}")

        # early stopping based on gmean
        cur_g = val_metrics.get('gmean', 0)
        if cur_g > best_g + 1e-8:
            best_g = cur_g
            best_state = {k: v.cpu() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if patience >= config.PATIENCE:
            print(f"Early stopping après epoch {epoch+1}, best_val_gmean={best_g:.4f}")
            break

    # restore best
    if best_state:
        model.load_state_dict(best_state)
    return model, history, val_metrics

def save_results(model, history, metrics, project, config, drive_base):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_id = f"{project}_{ts}"
    model_path = os.path.join(drive_base, "models", exp_id + ".pt")
    torch.save(model.state_dict(), model_path)
    results = {
        'experiment_id': exp_id,
        'project': project,
        'config': {
            'epochs': config.EPOCHS,
            'batch_size': config.BATCH_SIZE,
            'lr': config.LEARNING_RATE,
            'embedding_dim': config.EMBEDDING_DIM
        },
        'metrics': metrics,
        'history_len': len(history.get('train_loss', []))
    }
    with open(os.path.join(drive_base, "results", exp_id + ".json"), 'w') as f:
        json.dump(results, f, indent=4)
    # save history
    joblib.dump(history, os.path.join(drive_base, "results", exp_id + "_history.pkl"))
    print("Saved model & results to Drive:", model_path)
    return exp_id


In [ ]:
# Config
class Cfg:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 20
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2
    PATIENCE = 10

cfg = Cfg()

def run_intraproject_pipeline(project_name):
    print("=== Projet:", project_name, "===")
    metric_files = find_project_metric_files(project_name, PROMISE_METRICS_DIR)
    if not metric_files:
        print("Aucun fichier métriques trouvé pour", project_name); return None

    df_all = load_and_concat_metric_csvs(metric_files)
    if df_all.empty:
        print("Aucun contenu chargé."); return None

    # split n-1 -> n
    train_df, val_df = split_intraproject_by_version(df_all)
    if 'bug' not in train_df.columns:
        raise ValueError("Les CSV doivent contenir une colonne 'bug' binaire.")

    # prepare features
    X_train_raw, y_train_raw, feat_cols = prepare_features_and_labels(train_df)
    X_val_raw, y_val_raw, _ = prepare_features_and_labels(val_df, feature_cols=feat_cols)

    # outlier removal (EllipticEnvelope)
    X_train_clean, y_train_clean = remove_outliers_elliptic(X_train_raw, y_train_raw, contamination=0.05)

    # standardize
    X_train_s, X_val_s, scaler = standardize_train_val(X_train_clean, X_val_raw)

    # SMOTE
    sm = BorderlineSMOTE(random_state=42)
    try:
        X_train_bal, y_train_bal = sm.fit_resample(X_train_s, y_train_clean)
    except Exception as e:
        print("SMOTE failed, using unbalanced train:", e)
        X_train_bal, y_train_bal = X_train_s, y_train_clean

    print("Train size after resampling:", X_train_bal.shape, "Val size:", X_val_s.shape)

    # map metrics rows to code embeddings for both train and val
    # reconstruct dataframes corresponding to rows we kept (we lost indices after resampling)
    # -> easiest approach: rebuild mapping by nearest-by-feature-match (best effort)
    # We'll instead take original train_df rows matching cleaned X_train_clean by simple index positions if possible.
    # For robustness, map using the original cleaned indices from remove_outliers_elliptic logic - but we do not have them,
    # so we will fallback to map by original train_df rows order after operations.
    # Simpler: for reproducibility, map code embeddings using val_df and train_df pre-resampling (may have mismatch sizes).
    # We'll compute embeddings only for rows that we can match by filename columns.

    # Build source index (already built earlier as source_index)
    # Get embeddings for train_df and val_df using mapping where possible
    print("Embedding code files (peut être long).")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_code_embeds, train_matched = map_metrics_to_code_embeddings(train_df.reset_index(drop=True), source_index, tokenizer_code, model_code, device=device)
    val_code_embeds,   val_matched   = map_metrics_to_code_embeddings(val_df.reset_index(drop=True), source_index, tokenizer_code, model_code, device=device)

    # Filter out rows with missing code embeddings
    def filter_valid(X, code_embs, y, df_rows):
        Xf, Cf, yf = [], [], []
        for i, emb in enumerate(code_embs):
            if emb is None:
                continue
            # X may be standardized or raw; need to pick corresponding row
            # We'll use df_rows.iloc[i] mapping: get its standardized features using scaler
            Xf.append(X[i])
            Cf.append(emb)
            yf.append(int(y[i]))
        return np.array(Xf), np.array(Cf), np.array(yf)

    # For train: we used X_train_bal (post SMOTE), which may not align with original indices.
    # Simpler approach: recompute features for the subset of rows that have embeddings:
    # take rows from train_df that had embeddings (train_matched), compute their standardized features via scaler,
    # then optionally SMOTE the small set.
    # Implementation:
    train_df_reset = train_df.reset_index(drop=True)
    tr_idx_with_emb = [i for i, e in enumerate(train_code_embeds) if e is not None]
    if len(tr_idx_with_emb) < 10:
        print("Warning: trop peu de correspondances code pour le training:", len(tr_idx_with_emb))

    X_train_emb_rows = train_df_reset.loc[tr_idx_with_emb, feat_cols].fillna(0).values
    X_train_emb_rows_s = scaler.transform(X_train_emb_rows)  # use same scaler
    y_train_emb_rows = train_df_reset.loc[tr_idx_with_emb, 'bug'].astype(int).values
    code_train_embs = [train_code_embeds[i] for i in tr_idx_with_emb]

    # Apply SMOTE now (on the subset that has code embeddings)
    try:
        X_train_final, y_train_final = sm.fit_resample(X_train_emb_rows_s, y_train_emb_rows)
        # SMOTE doesn't produce code embeddings -> need to replicate code embeddings for synthetic samples.
        # This is a limitation: synthetic samples won't have code embeddings. Strategy: after SMOTE, keep only original samples (no augmentation) OR duplicate nearest code embeddings.
        # For scientific correctness, we'll **not** apply SMOTE to avoid synthetic samples without code. Instead, we'll undersample majority to balance.
        raise RuntimeError("Avoid SMOTE for combined metric+code since synthetic samples lack code embeddings.")
    except Exception as e:
        # fallback: undersample majority from available labeled rows
        print("SMOTE skipped for combined dataset: ", e)
        counts = Counter(y_train_emb_rows)
        maj = max(counts, key=counts.get)
        min_count = min(counts.values())
        # sample indices per class
        idxs = []
        for lab in [0,1]:
            ids = np.where(y_train_emb_rows==lab)[0]
            if len(ids) > min_count:
                ids = np.random.choice(ids, min_count, replace=False)
            idxs.extend(list(ids))
        idxs = np.array(idxs)
        X_train_final = X_train_emb_rows_s[idxs]
        y_train_final = y_train_emb_rows[idxs]
        code_train_final = np.array([code_train_embs[i] for i in range(len(code_train_embs))])[idxs]

    # Prepare validation set: filter only rows with code embeddings
    val_df_reset = val_df.reset_index(drop=True)
    val_idx_with_emb = [i for i,e in enumerate(val_code_embeds) if e is not None]
    if len(val_idx_with_emb) == 0:
        raise RuntimeError("Aucune correspondance code pour la validation du projet %s" % project_name)
    X_val_rows = val_df_reset.loc[val_idx_with_emb, feat_cols].fillna(0).values
    X_val_rows_s = scaler.transform(X_val_rows)
    y_val_rows = val_df_reset.loc[val_idx_with_emb, 'bug'].astype(int).values
    code_val_final = np.array([val_code_embeds[i] for i in val_idx_with_emb])

    # Now ensure X_train_final, code_train_final, y_train_final exist
    # We already created code_train_final above
    if 'code_train_final' not in locals():
        # Build from available code_train_embs and selected idxs
        code_train_final = np.array([code_train_embs[i] for i in idxs])

    print("Final dataset sizes: train:", X_train_final.shape, "val:", X_val_rows_s.shape)

    # build datasets and dataloaders
    train_dataset = MetricsCodeDataset(X_train_final, code_train_final, y_train_final)
    val_dataset   = MetricsCodeDataset(X_val_rows_s, code_val_final, y_val_rows)
    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

    # instantiate model
    metrics_dim = X_train_final.shape[1]
    code_emb_dim = code_train_final.shape[1]
    model = FusionModel(metrics_dim, code_emb_dim, cfg).to(cfg.DEVICE)

    # train
    model_trained, history, val_metrics = train_model(model, train_loader, val_loader, cfg, cfg.DEVICE, DRIVE_BASE)

    # save
    exp_id = save_results(model_trained, history, val_metrics, project_name, cfg, DRIVE_BASE)
    return exp_id, val_metrics, history

# Exemple d'appel pour un projet (ex: ant)
run_intraproject_pipeline("ant")


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter
import re

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    # Nouveaux paramètres pour l'alignement
    MAX_CODE_LENGTH = 512
    CROSS_ATTENTION_HEADS = 8

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé avec alignement
class AlignedBugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels, file_names):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)
        self.file_names = file_names

        # Vérifications de cohérence
        assert len(self.metrics) == len(self.labels) == len(self.file_names), "Mismatch dans les données alignées"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx], self.file_names[idx]

# Modèle avec alignement par attention croisée avancée
class CrossAttentionAlignedModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()

        # Projection des métriques
        self.metric_projection = nn.Sequential(
            nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.ReLU(),  # ← CHANGEMENT: ReLU au lieu de GELU
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder pour les métriques
        self.metrics_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=Config.EMBEDDING_DIM,
                nhead=Config.N_HEADS,
                dropout=Config.DROPOUT,
                activation='relu',  # ← CHANGEMENT: ReLU au lieu de gelu
                batch_first=True,
                norm_first=True
            ),
            num_layers=Config.NUM_LAYERS
        )

        # Modèle pré-entraîné pour le code
        self.code_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        self.code_projection = nn.Linear(768, Config.EMBEDDING_DIM)

        # ATTENTION CROISÉE BIDIRECTIONNELLE
        # Métriques → Code
        self.metrics_to_code_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.CROSS_ATTENTION_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Code → Métriques
        self.code_to_metrics_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.CROSS_ATTENTION_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion multi-modale
        self.fusion_layer = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 3, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),  # ← CHANGEMENT: ReLU au lieu de GELU
            nn.Dropout(Config.DROPOUT),
            nn.Linear(Config.EMBEDDING_DIM, Config.EMBEDDING_DIM // 2),
            nn.ReLU()   # ← CHANGEMENT: ReLU au lieu de GELU
        )

        # Classifier avec features alignées
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM // 2, 256),
            nn.LayerNorm(256),
            nn.ReLU(),  # ← CHANGEMENT: ReLU au lieu de GELU
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),  # ← CHANGEMENT: ReLU au lieu de GELU
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        batch_size = metrics.size(0)

        # Encoder les métriques
        metrics_proj = self.metric_projection(metrics).unsqueeze(1)  # [B, 1, D]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, D]

        # Encoder le code
        code_outputs = self.code_encoder(**code_tokens)
        code_embeddings = self.code_projection(code_outputs.last_hidden_state)  # [B, L, D]

        # ATTENTION CROISÉE BIDIRECTIONNELLE
        # 1. Métriques → Code (les métriques interrogent le code)
        metrics_attended_code, _ = self.metrics_to_code_attention(
            query=metrics_encoded,    # [B, 1, D]
            key=code_embeddings,      # [B, L, D]
            value=code_embeddings     # [B, L, D]
        )  # [B, 1, D]

        # 2. Code → Métriques (le code interroge les métriques)
        # Prendre la moyenne des embeddings de code comme query
        code_mean = code_embeddings.mean(dim=1, keepdim=True)  # [B, 1, D]
        code_attended_metrics, _ = self.code_to_metrics_attention(
            query=code_mean,          # [B, 1, D]
            key=metrics_encoded,      # [B, 1, D]
            value=metrics_encoded     # [B, 1, D]
        )  # [B, 1, D]

        # FUSION MULTI-SOURCE
        combined_features = torch.cat([
            metrics_encoded[:, 0, :],           # Représentation originale des métriques
            metrics_attended_code[:, 0, :],     # Métriques enrichies par le code
            code_attended_metrics[:, 0, :]      # Code enrichi par les métriques
        ], dim=-1)  # [B, 3*D]

        # Fusion finale
        fused = self.fusion_layer(combined_features)  # [B, D/2]

        # Classification
        logits = self.classifier(fused)

        return logits

# Modèle avec alignement par attention hiérarchique
class HierarchicalAlignedModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()

        # Encoder pour métriques avec plus de capacité
        self.metric_encoder = nn.Sequential(
            nn.Linear(input_metric_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),  # ← CHANGEMENT: ReLU au lieu de GELU
            nn.Dropout(Config.DROPOUT),
            nn.Linear(512, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM)
        )

        # Code encoder avec RoBERTa
        self.code_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        self.code_adapter = nn.Linear(768, Config.EMBEDDING_DIM)

        # Alignement par similarité cosinus
        self.alignment_layer = nn.CosineSimilarity(dim=-1)

        # Transformer de fusion
        self.fusion_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=Config.EMBEDDING_DIM,
                nhead=Config.N_HEADS,
                dropout=Config.DROPOUT,
                batch_first=True
            ),
            num_layers=2
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Encoder métriques
        metrics_emb = self.metric_encoder(metrics).unsqueeze(1)  # [B, 1, D]

        # Encoder code
        code_outputs = self.code_encoder(**code_tokens)
        code_emb = self.code_adapter(code_outputs.last_hidden_state)  # [B, L, D]

        # Alignement par attention
        alignment_scores = torch.bmm(metrics_emb, code_emb.transpose(1, 2))  # [B, 1, L]
        alignment_weights = torch.softmax(alignment_scores, dim=-1)

        # Code aligné avec les métriques
        aligned_code = torch.bmm(alignment_weights, code_emb)  # [B, 1, D]

        # Fusion
        combined = torch.cat([metrics_emb, aligned_code], dim=1)  # [B, 2, D]
        fused = self.fusion_transformer(combined)

        # Classification sur la représentation moyenne
        pooled = fused.mean(dim=1)
        return self.classifier(pooled)

# FONCTIONS D'ALIGNEMENT AMÉLIORÉES
def extract_filename_from_path(file_path):
    """Extrait le nom de fichier sans extension"""
    base_name = os.path.basename(file_path)
    name_without_ext = os.path.splitext(base_name)[0]
    # Nettoyer le nom de fichier
    cleaned_name = re.sub(r'[^a-zA-Z0-9_]', '', name_without_ext)
    return cleaned_name.lower()

def align_metrics_with_code(df_metrics, code_files_dict):
    """Aligne explicitement les métriques avec le code source correspondant"""
    aligned_metrics = []
    aligned_code = []
    aligned_labels = []
    aligned_filenames = []

    matches_count = 0
    partial_matches_count = 0
    no_matches_count = 0

    for idx, row in df_metrics.iterrows():
        # Extraire le nom de fichier des métriques
        if 'name' in row:
            metric_filename = extract_filename_from_path(str(row['name']))
        else:
            metric_filename = f"file_{idx}"

        # Chercher le code correspondant
        matching_code = None
        match_type = "none"

        # 1. Recherche exacte
        for code_filename, code_content in code_files_dict.items():
            code_base = extract_filename_from_path(code_filename)
            if code_base == metric_filename:
                matching_code = code_content
                match_type = "exact"
                matches_count += 1
                break

        # 2. Recherche partielle
        if matching_code is None:
            for code_filename, code_content in code_files_dict.items():
                code_base = extract_filename_from_path(code_filename)
                if (metric_filename in code_base or code_base in metric_filename or
                    code_base.replace('_', '').replace('-', '') == metric_filename.replace('_', '').replace('-', '')):
                    matching_code = code_content
                    match_type = "partial"
                    partial_matches_count += 1
                    break

        # 3. Fallback stratégique
        if matching_code is None and code_files_dict:
            # Utiliser le fichier avec le nom le plus similaire
            matching_code = next(iter(code_files_dict.values()))
            match_type = "fallback"
            no_matches_count += 1

        if matching_code is not None:
            # Préparer les métriques (exclure 'name' et 'bug')
            metric_data = {}
            for col, value in row.items():
                if col not in ['name', 'bug'] and pd.api.types.is_numeric_dtype(type(value)):
                    metric_data[col] = value

            aligned_metrics.append(metric_data)
            aligned_code.append(matching_code)
            aligned_labels.append(1 if row.get('bug', 0) > 0 else 0)
            aligned_filenames.append(f"{metric_filename}_{match_type}")

    print(f"✓ Alignement: {matches_count} exacts, {partial_matches_count} partiels, {no_matches_count} fallbacks")
    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

def load_and_align_data(metrics_path, code_path):
    """Charge et aligne les métriques avec le code source"""
    # Charger les métriques
    df_metrics = pd.read_csv(metrics_path)

    # Charger tous les fichiers code dans un dictionnaire
    code_files_dict = {}
    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        file_path = os.path.join(root, file)
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            code_content = f.read().strip()
                            if code_content:
                                code_files_dict[file] = code_content
                    except Exception as e:
                        print(f"Erreur lecture {file}: {e}")
                        continue

    print(f"✓ {len(code_files_dict)} fichiers code chargés")

    # Aligner les données
    aligned_metrics, aligned_code, aligned_labels, aligned_filenames = align_metrics_with_code(
        df_metrics, code_files_dict
    )

    print(f"✓ {len(aligned_metrics)} échantillons alignés sur {len(df_metrics)} métriques originales")

    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

# FONCTIONS DE PRÉTRAITEMENT AVEC ALIGNEMENT
def preprocess_aligned_metrics(metrics_list, labels):
    """Prétraite les métriques alignées AVEC les labels"""
    if not metrics_list:
        return np.array([]), np.array([]), None

    df_metrics = pd.DataFrame(metrics_list)

    # Gérer les valeurs manquantes
    if df_metrics.isnull().any().any():
        df_metrics = df_metrics.fillna(df_metrics.median())

    # Supprimer les features à variance nulle
    var_threshold = 0.01
    low_var_cols = df_metrics.var() < var_threshold
    if low_var_cols.any():
        df_metrics = df_metrics.loc[:, ~low_var_cols]

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(df_metrics)

    return features_scaled, np.array(labels), scaler

# FONCTIONS UTILITAIRES EXISTANTES (adaptées)
def apply_borderline_smote(X, y, filenames):
    """Version adaptée pour garder les noms de fichiers"""
    print(f"Distribution avant SMOTE: {Counter(y)}")

    # Vérifier qu'il y a assez d'échantillons minoritaires
    min_samples = min(Counter(y).values())
    k_neighbors = min(3, min_samples - 1) if min_samples > 1 else 1

    try:
        smote = BorderlineSMOTE(random_state=42, k_neighbors=k_neighbors, kind='borderline-1')
        X_resampled, y_resampled = smote.fit_resample(X, y)
        print("✓ BorderlineSMOTE appliqué avec succès")
    except Exception as e:
        print(f"Erreur BorderlineSMOTE: {e}, utilisation de SMOTE standard")
        smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_resampled, y_resampled = smote.fit_resample(X, y)

    # Adapter les filenames
    n_original = len(filenames)
    n_resampled = len(X_resampled)

    if n_resampled > n_original:
        # Répliquer les filenames existants
        indices_map = list(range(n_original)) + \
                     list(np.random.randint(0, n_original, n_resampled - n_original))
        filenames_resampled = [filenames[i] for i in indices_map]
    else:
        filenames_resampled = filenames[:n_resampled]

    print(f"Distribution après SMOTE: {Counter(y_resampled)}")
    return X_resampled, y_resampled, filenames_resampled

def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    return weight_tensor

def prepare_aligned_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset avec alignement explicite"""
    # Charger et aligner les données
    metrics_list, code_lines, labels, filenames = load_and_align_data(metrics_path, code_path)

    if not metrics_list:
        raise ValueError("Aucune donnée alignée trouvée")

    # Prétraiter les métriques
    features, labels, scaler = preprocess_aligned_metrics(metrics_list, labels)

    # Équilibrer les classes si demandé
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, filenames = apply_borderline_smote(features, labels, filenames)
            print("✓ Données équilibrées après alignement")
        except Exception as e:
            print(f"Erreur équilibrage: {e}")

    # Tokeniser le code
    code_tokens = tokenizer(
        code_lines,
        padding=True,
        truncation=True,
        return_tensors='pt',
        max_length=Config.MAX_CODE_LENGTH
    )

    return features, code_tokens, labels, scaler, filenames

# FONCTIONS D'ÉVALUATION
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels, _ in dataloader:  # ← Ajout _ pour filenames
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# FONCTION D'ENTRAÎNEMENT AVEC ALIGNEMENT
def train_aligned_model(model_type="cross_attention"):
    """Entraîne un modèle avec alignement explicite - VERSION CORRIGÉE"""
    print(f"\n{'='*60}")
    print(f"ENTRAÎNEMENT AVEC ALIGNEMENT: {model_type.upper()}")
    print(f"{'='*60}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Chemins des données
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/xerces/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/xerces/"

    train_versions = ["1.1", "1.2", "1.3"]
    val_version = "1.4.4"

    # Initialisation correcte des listes
    all_train_features = []
    all_train_labels = []
    all_train_filenames = []
    all_train_code_tokens = {'input_ids': [], 'attention_mask': []}

    # Charger les données d'entraînement
    for version in train_versions:
        metrics_path = f"{base_metrics_path}xerces-{version}.csv"
        code_path = f"{base_code_path}xerces-{version}"

        try:
            features, code_tokens, labels, _, filenames = prepare_aligned_dataset(
                metrics_path, code_path, tokenizer, apply_balancing=False
            )

            all_train_features.append(features)
            all_train_labels.extend(labels)
            all_train_filenames.extend(filenames)

            # Stocker les tokens
            all_train_code_tokens['input_ids'].append(code_tokens['input_ids'])
            all_train_code_tokens['attention_mask'].append(code_tokens['attention_mask'])

            print(f"✓ Version {version}: {len(features)} échantillons alignés")

        except Exception as e:
            print(f"✗ Erreur avec la version {version}: {e}")
            continue

    # Vérifier qu'on a des données
    if not all_train_features:
        raise ValueError("Aucune donnée d'entraînement chargée")

    # Concaténer toutes les données d'entraînement
    X_train = np.vstack(all_train_features)
    y_train = np.array(all_train_labels)

    # Concaténer les tokens
    all_train_code = {
        'input_ids': torch.cat(all_train_code_tokens['input_ids']),
        'attention_mask': torch.cat(all_train_code_tokens['attention_mask'])
    }

    # Équilibrer SI nécessaire
    apply_balancing = True
    if apply_balancing and len(set(y_train)) > 1:
        print("Application de BorderlineSMOTE sur l'ensemble d'entraînement...")
        X_train, y_train, all_train_filenames = apply_borderline_smote(
            X_train, y_train, all_train_filenames
        )
        print("⚠️ Note: Les tokens de code ne sont pas réorganisés après SMOTE")

    # Charger les données de validation
    print("Chargement des données de validation...")
    metrics_val_path = f"{base_metrics_path}xerces-{val_version}.csv"
    code_val_path = f"{base_code_path}xerces-{val_version}"

    X_val, code_val_tokens, y_val, _, val_filenames = prepare_aligned_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    # Vérifier les dimensions
    print(f"Données d'entraînement: {X_train.shape[0]} échantillons, {X_train.shape[1]} features")
    print(f"Données de validation: {X_val.shape[0]} échantillons")

    # Créer les datasets
    train_ds = AlignedBugDataset(X_train, all_train_code, y_train, all_train_filenames)
    val_ds = AlignedBugDataset(X_val, code_val_tokens, y_val, val_filenames)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Passer la dimension exacte
    input_dim = X_train.shape[1]
    if model_type == "hierarchical":
        model = HierarchicalAlignedModel(input_metric_dim=input_dim).to(Config.DEVICE)
    else:
        model = CrossAttentionAlignedModel(input_metric_dim=input_dim).to(Config.DEVICE)

    print(f"✓ Modèle {model_type} initialisé avec {input_dim} features")

    # Optimiseur et loss
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    optimizer = torch.optim.AdamW([
        {'params': model.code_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
        {'params': [p for n, p in model.named_parameters() if 'code_encoder' not in n], 'lr': Config.LEARNING_RATE}
    ], weight_decay=0.01)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0

    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels, _) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds) if train_preds else 1
        train_acc = accuracy_score(train_labels_list, train_preds) if train_labels_list else 0
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss if val_loss else 0)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
        print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            joblib.dump(cm, f"{Config.SAVE_PATH}conf_matrix_{model_type}.pkl")
            joblib.dump(val_preds, f"{Config.SAVE_PATH}val_preds_{model_type}.pkl")
            joblib.dump(val_labels, f"{Config.SAVE_PATH}val_labels_{model_type}.pkl")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 10:
            print("Early stopping déclenché")
            break

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Visualisations
    plt.figure(figsize=(15, 5))

    # Courbe 1: Loss
    plt.subplot(1, 3, 1)
    epochs_range = range(1, len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
    plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Courbe 2: Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
    plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Courbe 3: F1-score et G-mean
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, f1_scores, 'g-', label='F1-Score')
    plt.plot(epochs_range, gmean_scores, 'm-', label='G-Mean')
    plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}training_curves_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

# FONCTION PRINCIPALE
def main():
    """Fonction principale pour lancer l'entraînement avec alignement"""
    print("🚀 DÉMARRAGE AVEC ALIGNEMENT EXPLICITE MÉTRIQUES-CODE")
    print("Utilisation de l'attention croisée pour l'alignement multi-modal")
    print("Activation: ReLU (au lieu de GELU)")

    try:
        # Entraîner les différents modèles avec alignement
        print("\n1. Modèle avec attention croisée bidirectionnelle")
        model_cross, results_cross = train_aligned_model("cross_attention")

        print("\n2. Modèle avec alignement hiérarchique")
        model_hierarchical, results_hierarchical = train_aligned_model("hierarchical")

        # Comparaison des résultats
        print("\n" + "="*60)
        print("COMPARAISON DES MODÈLES AVEC ALIGNEMENT")
        print("="*60)

        print(f"{'Métrique':<20} {'Cross-Attention':<15} {'Hierarchical':<15} {'Meilleur':<15}")
        print("-" * 65)

        metrics = ['F1-Score', 'G-Mean', 'Accuracy', 'Sensitivity', 'Specificity']
        cross_scores = [
            results_cross['best_f1'], results_cross['final_gmean'],
            results_cross['final_accuracy'], results_cross['final_sensitivity'],
            results_cross['final_specificity']
        ]
        hierarchical_scores = [
            results_hierarchical['best_f1'], results_hierarchical['final_gmean'],
            results_hierarchical['final_accuracy'], results_hierarchical['final_sensitivity'],
            results_hierarchical['final_specificity']
        ]

        for metric, cross, hier in zip(metrics, cross_scores, hierarchical_scores):
            better = "Cross" if cross > hier else "Hierarchical" if hier > cross else "Égalité"
            print(f"{metric:<20} {cross:<15.4f} {hier:<15.4f} {better:<15}")

        # Recommandation finale
        best_model = "Cross-Attention" if results_cross['best_f1'] > results_hierarchical['best_f1'] else "Hierarchical"
        print(f"\n✓ Modèle recommandé: {best_model}")

    except Exception as e:
        print(f"❌ Erreur lors de l'entraînement: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter
import re

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    # Nouveaux paramètres pour l'alignement
    MAX_CODE_LENGTH = 512
    CROSS_ATTENTION_HEADS = 8

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé avec alignement
class AlignedBugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels, file_names):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)
        self.file_names = file_names

        # Vérifications de cohérence
        assert len(self.metrics) == len(self.labels) == len(self.file_names), "Mismatch dans les données alignées"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx], self.file_names[idx]

# Modèle avec alignement par attention croisée avancée
class CrossAttentionAlignedModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()

        # Projection des métriques
        self.metric_projection = nn.Sequential(
            nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder pour les métriques
        self.metrics_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=Config.EMBEDDING_DIM,
                nhead=Config.N_HEADS,
                dropout=Config.DROPOUT,
                activation='relu',
                batch_first=True,
                norm_first=True
            ),
            num_layers=Config.NUM_LAYERS
        )

        # Modèle pré-entraîné pour le code
        self.code_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        self.code_projection = nn.Linear(768, Config.EMBEDDING_DIM)

        # ATTENTION CROISÉE BIDIRECTIONNELLE
        # Métriques → Code
        self.metrics_to_code_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.CROSS_ATTENTION_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Code → Métriques
        self.code_to_metrics_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.CROSS_ATTENTION_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion multi-modale
        self.fusion_layer = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 3, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(Config.EMBEDDING_DIM, Config.EMBEDDING_DIM // 2),
            nn.ReLU()
        )

        # Classifier avec features alignées
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM // 2, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        batch_size = metrics.size(0)

        # Encoder les métriques
        metrics_proj = self.metric_projection(metrics).unsqueeze(1)  # [B, 1, D]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, D]

        # Encoder le code
        code_outputs = self.code_encoder(**code_tokens)
        code_embeddings = self.code_projection(code_outputs.last_hidden_state)  # [B, L, D]

        # ATTENTION CROISÉE BIDIRECTIONNELLE
        # 1. Métriques → Code (les métriques interrogent le code)
        metrics_attended_code, _ = self.metrics_to_code_attention(
            query=metrics_encoded,    # [B, 1, D]
            key=code_embeddings,      # [B, L, D]
            value=code_embeddings     # [B, L, D]
        )  # [B, 1, D]

        # 2. Code → Métriques (le code interroge les métriques)
        # Prendre la moyenne des embeddings de code comme query
        code_mean = code_embeddings.mean(dim=1, keepdim=True)  # [B, 1, D]
        code_attended_metrics, _ = self.code_to_metrics_attention(
            query=code_mean,          # [B, 1, D]
            key=metrics_encoded,      # [B, 1, D]
            value=metrics_encoded     # [B, 1, D]
        )  # [B, 1, D]

        # FUSION MULTI-SOURCE
        combined_features = torch.cat([
            metrics_encoded[:, 0, :],           # Représentation originale des métriques
            metrics_attended_code[:, 0, :],     # Métriques enrichies par le code
            code_attended_metrics[:, 0, :]      # Code enrichi par les métriques
        ], dim=-1)  # [B, 3*D]

        # Fusion finale
        fused = self.fusion_layer(combined_features)  # [B, D/2]

        # Classification
        logits = self.classifier(fused)

        return logits

# Modèle avec alignement par attention hiérarchique
class HierarchicalAlignedModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()

        # Encoder pour métriques avec plus de capacité
        self.metric_encoder = nn.Sequential(
            nn.Linear(input_metric_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(512, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM)
        )

        # Code encoder avec RoBERTa
        self.code_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        self.code_adapter = nn.Linear(768, Config.EMBEDDING_DIM)

        # Alignement par similarité cosinus
        self.alignment_layer = nn.CosineSimilarity(dim=-1)

        # Transformer de fusion
        self.fusion_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=Config.EMBEDDING_DIM,
                nhead=Config.N_HEADS,
                dropout=Config.DROPOUT,
                batch_first=True
            ),
            num_layers=2
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Encoder métriques
        metrics_emb = self.metric_encoder(metrics).unsqueeze(1)  # [B, 1, D]

        # Encoder code
        code_outputs = self.code_encoder(**code_tokens)
        code_emb = self.code_adapter(code_outputs.last_hidden_state)  # [B, L, D]

        # Alignement par attention
        alignment_scores = torch.bmm(metrics_emb, code_emb.transpose(1, 2))  # [B, 1, L]
        alignment_weights = torch.softmax(alignment_scores, dim=-1)

        # Code aligné avec les métriques
        aligned_code = torch.bmm(alignment_weights, code_emb)  # [B, 1, D]

        # Fusion
        combined = torch.cat([metrics_emb, aligned_code], dim=1)  # [B, 2, D]
        fused = self.fusion_transformer(combined)

        # Classification sur la représentation moyenne
        pooled = fused.mean(dim=1)
        return self.classifier(pooled)

# FONCTIONS D'ALIGNEMENT AMÉLIORÉES
def extract_filename_from_path(file_path):
    """Extrait le nom de fichier avec plus de flexibilité"""
    if pd.isna(file_path) or file_path is None:
        return []

    base_name = os.path.basename(str(file_path))
    name_without_ext = os.path.splitext(base_name)[0]

    # Nettoyage plus agressif pour trouver des correspondances
    cleaned_name = re.sub(r'[^a-zA-Z0-9]', '', name_without_ext.lower())

    if not cleaned_name:
        return []

    # Essayer plusieurs formats
    variations = set()
    variations.add(cleaned_name)
    variations.add(cleaned_name.replace('test', '').replace('testcase', ''))
    variations.add(re.sub(r'\d+', '', cleaned_name))  # Supprimer les chiffres

    if '_' in cleaned_name:
        variations.add(cleaned_name.split('_')[0])
    if '.' in cleaned_name:
        variations.add(cleaned_name.split('.')[0])

    # Ajouter des variations avec et sans underscores/chiffres
    variations.add(cleaned_name.replace('_', ''))
    variations.add(re.sub(r'[^a-zA-Z]', '', cleaned_name))

    # Filtrer les chaînes vides
    return [v for v in variations if v and len(v) > 1]

def align_metrics_with_code(df_metrics, code_files_dict):
    """Aligne avec des critères plus flexibles"""
    aligned_metrics = []
    aligned_code = []
    aligned_labels = []
    aligned_filenames = []

    matches_count = 0
    partial_matches_count = 0
    no_matches_count = 0

    # Créer un index des fichiers code pour recherche rapide
    code_index = {}
    for code_filename, code_content in code_files_dict.items():
        code_variations = extract_filename_from_path(code_filename)
        for variation in code_variations:
            if variation:  # Éviter les chaînes vides
                code_index[variation] = (code_content, code_filename)

    print(f"✓ Index créé avec {len(code_index)} variations de noms de code")

    for idx, row in df_metrics.iterrows():
        # Extraire le nom de fichier des métriques
        if 'name' in row:
            metric_variations = extract_filename_from_path(str(row['name']))
        else:
            metric_variations = [f"file{idx}"]

        # Chercher le code correspondant
        matching_code = None
        match_type = "none"
        matched_filename = ""

        # Recherche exacte dans toutes les variations
        for metric_var in metric_variations:
            if metric_var in code_index:
                matching_code, matched_filename = code_index[metric_var]
                match_type = "exact"
                matches_count += 1
                break

        # Recherche partielle si pas de correspondance exacte
        if matching_code is None:
            for metric_var in metric_variations:
                for code_var, (code_content, filename) in code_index.items():
                    if (metric_var and code_var and
                        (metric_var in code_var or code_var in metric_var)):
                        # Vérifier que c'est une correspondance significative
                        if len(metric_var) > 3 and len(code_var) > 3:
                            matching_code = code_content
                            matched_filename = filename
                            match_type = "partial"
                            partial_matches_count += 1
                            break
                if matching_code:
                    break

        # Dernier fallback : utiliser le fichier le plus proche
        if matching_code is None and code_files_dict:
            # Essayer de trouver le fichier avec le nom le plus similaire
            best_match = None
            best_score = 0

            for metric_var in metric_variations:
                for code_var, (code_content, filename) in code_index.items():
                    # Calcul de similarité simple
                    score = 0
                    if metric_var in code_var:
                        score = len(metric_var) / len(code_var)
                    elif code_var in metric_var:
                        score = len(code_var) / len(metric_var)

                    if score > best_score:
                        best_score = score
                        best_match = (code_content, filename)

            if best_match and best_score > 0.3:  # Seuil de similarité
                matching_code, matched_filename = best_match
                match_type = "similar"
            else:
                matching_code, matched_filename = next(iter(code_index.values()))
                match_type = "fallback"
            no_matches_count += 1

        if matching_code is not None:
            # Préparer les métriques (exclure 'name' et 'bug')
            metric_data = {}
            for col, value in row.items():
                if col not in ['name', 'bug'] and pd.api.types.is_numeric_dtype(type(value)):
                    metric_data[col] = float(value) if pd.notna(value) else 0.0

            aligned_metrics.append(metric_data)
            aligned_code.append(matching_code)
            aligned_labels.append(1 if row.get('bug', 0) > 0 else 0)
            aligned_filenames.append(f"{matched_filename}_{match_type}")

    print(f"✓ Alignement: {matches_count} exacts, {partial_matches_count} partiels, {no_matches_count} fallbacks/similaires")
    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

def load_and_align_data(metrics_path, code_path):
    """Charge et aligne les métriques avec le code source"""
    # Charger les métriques
    df_metrics = pd.read_csv(metrics_path)
    print(f"✓ {len(df_metrics)} métriques chargées depuis {os.path.basename(metrics_path)}")

    # Charger tous les fichiers code dans un dictionnaire
    code_files_dict = {}
    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        file_path = os.path.join(root, file)
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            code_content = f.read().strip()
                            if code_content and len(code_content) > 10:  # Fichier non vide
                                code_files_dict[file] = code_content
                    except Exception as e:
                        print(f"Erreur lecture {file}: {e}")
                        continue

    print(f"✓ {len(code_files_dict)} fichiers code chargés depuis {os.path.basename(code_path)}")

    if not code_files_dict:
        print("⚠️ Aucun fichier code trouvé!")
        return [], [], [], []

    # Aligner les données
    aligned_metrics, aligned_code, aligned_labels, aligned_filenames = align_metrics_with_code(
        df_metrics, code_files_dict
    )

    print(f"✓ {len(aligned_metrics)} échantillons alignés sur {len(df_metrics)} métriques originales")

    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

# FONCTIONS DE PRÉTRAITEMENT AVEC ALIGNEMENT
def preprocess_aligned_metrics(metrics_list, labels):
    """Prétraite les métriques alignées AVEC les labels"""
    if not metrics_list:
        return np.array([]), np.array([]), None

    df_metrics = pd.DataFrame(metrics_list)

    # Gérer les valeurs manquantes
    if df_metrics.isnull().any().any():
        df_metrics = df_metrics.fillna(df_metrics.median())

    # Supprimer les features à variance nulle
    var_threshold = 0.01
    variances = df_metrics.var()
    low_var_cols = variances < var_threshold
    if low_var_cols.any():
        df_metrics = df_metrics.loc[:, ~low_var_cols]
        print(f"✓ {low_var_cols.sum()} colonnes à faible variance supprimées")

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(df_metrics)

    print(f"✓ Métriques prétraitées: {features_scaled.shape}")
    return features_scaled, np.array(labels), scaler

# FONCTIONS UTILITAIRES CORRIGÉES
def apply_smote_with_realignment(X, y, code_tokens):
    """Applique SMOTE et réaligne correctement les tokens de code"""
    print(f"Distribution avant SMOTE: {Counter(y)}")

    original_size = len(X)

    # Appliquer SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)

    new_size = len(X_resampled)
    print(f"Distribution après SMOTE: {Counter(y_resampled)}")

    # Réaligner les tokens de code
    if new_size <= original_size:
        # Sous-échantillonnage : prendre les premiers
        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][:new_size],
            'attention_mask': code_tokens['attention_mask'][:new_size]
        }
    else:
        # Sur-échantillonnage : dupliquer aléatoirement
        indices = list(range(original_size)) + \
                 list(np.random.randint(0, original_size, new_size - original_size))

        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][indices],
            'attention_mask': code_tokens['attention_mask'][indices]
        }

    print(f"✓ Tokens réalignés: {original_size} → {new_size} échantillons")
    return X_resampled, y_resampled, code_tokens_resampled

def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    print(f"✓ Poids des classes: {weights}")
    return weight_tensor

def prepare_aligned_dataset_with_smote(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset avec alignement explicite et gestion correcte de SMOTE"""
    # Charger et aligner les données
    metrics_list, code_lines, labels, filenames = load_and_align_data(metrics_path, code_path)

    if not metrics_list:
        raise ValueError("Aucune donnée alignée trouvée")

    # Prétraiter les métriques
    features, labels, scaler = preprocess_aligned_metrics(metrics_list, labels)

    # Tokeniser le code AVANT SMOTE
    original_code_tokens = tokenizer(
        code_lines,
        padding=True,
        truncation=True,
        return_tensors='pt',
        max_length=Config.MAX_CODE_LENGTH
    )

    # Équilibrer les classes SI demandé avec réalignement correct
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_tokens = apply_smote_with_realignment(
                features, labels, original_code_tokens
            )
            print("✓ SMOTE appliqué avec réalignement correct des tokens")
            return features, code_tokens, labels, scaler, filenames

        except Exception as e:
            print(f"Erreur équilibrage: {e}, utilisation des données originales")

    return features, original_code_tokens, labels, scaler, filenames

# FONCTIONS D'ÉVALUATION
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels, _ in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# FONCTION D'ENTRAÎNEMENT CORRIGÉE
def train_aligned_model(model_type="cross_attention"):
    """Entraîne un modèle avec alignement explicite - VERSION CORRIGÉE"""
    print(f"\n{'='*60}")
    print(f"ENTRAÎNEMENT AVEC ALIGNEMENT: {model_type.upper()}")
    print(f"{'='*60}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Chemins des données
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/xerces/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/xerces/"

    train_versions = ["1.1", "1.2", "1.3"]
    val_version = "1.4.4"

    # Initialisation correcte des listes
    all_train_features = []
    all_train_labels = []
    all_train_filenames = []
    all_train_code_tokens = {'input_ids': [], 'attention_mask': []}

    # Charger les données d'entraînement AVEC SMOTE INDIVIDUEL
    for version in train_versions:
        metrics_path = f"{base_metrics_path}xerces-{version}.csv"
        code_path = f"{base_code_path}xerces-{version}"

        try:
            # Appliquer SMOTE individuellement sur chaque version
            features, code_tokens, labels, _, filenames = prepare_aligned_dataset_with_smote(
                metrics_path, code_path, tokenizer, apply_balancing=True  # SMOTE individuel
            )

            all_train_features.append(features)
            all_train_labels.extend(labels)
            all_train_filenames.extend(filenames)

            all_train_code_tokens['input_ids'].append(code_tokens['input_ids'])
            all_train_code_tokens['attention_mask'].append(code_tokens['attention_mask'])

            print(f"✓ Version {version}: {len(features)} échantillons alignés et équilibrés")

        except Exception as e:
            print(f"✗ Erreur avec la version {version}: {e}")
            continue

    # Vérifier qu'on a des données
    if not all_train_features:
        raise ValueError("Aucune donnée d'entraînement chargée")

    # Concaténer toutes les données d'entraînement (SANS SMOTE GLOBAL)
    X_train = np.vstack(all_train_features)
    y_train = np.array(all_train_labels)

    # Concaténer les tokens
    all_train_code = {
        'input_ids': torch.cat(all_train_code_tokens['input_ids']),
        'attention_mask': torch.cat(all_train_code_tokens['attention_mask'])
    }

    print(f"✓ Données d'entraînement finales: {X_train.shape[0]} échantillons")
    print(f"✓ Distribution: {Counter(y_train)}")
    print(f"✓ Tokens de code: {all_train_code['input_ids'].shape}")

    # Charger les données de validation (SANS SMOTE)
    print("\nChargement des données de validation...")
    metrics_val_path = f"{base_metrics_path}xerces-{val_version}.csv"
    code_val_path = f"{base_code_path}xerces-{val_version}"

    X_val, code_val_tokens, y_val, _, val_filenames = prepare_aligned_dataset_with_smote(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False  # Pas de SMOTE pour la validation
    )

    print(f"✓ Données de validation: {X_val.shape[0]} échantillons")
    print(f"✓ Distribution validation: {Counter(y_val)}")

    # Vérifier la cohérence des dimensions
    assert X_train.shape[1] == X_val.shape[1], f"Mismatch features: train{X_train.shape[1]} vs val{X_val.shape[1]}"

    # Créer les datasets
    train_ds = AlignedBugDataset(X_train, all_train_code, y_train, all_train_filenames)
    val_ds = AlignedBugDataset(X_val, code_val_tokens, y_val, val_filenames)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Initialiser le modèle avec la dimension exacte
    input_dim = X_train.shape[1]
    if model_type == "hierarchical":
        model = HierarchicalAlignedModel(input_metric_dim=input_dim).to(Config.DEVICE)
    else:
        model = CrossAttentionAlignedModel(input_metric_dim=input_dim).to(Config.DEVICE)

    print(f"✓ Modèle {model_type} initialisé avec {input_dim} features")

    # Optimiseur et loss
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    optimizer = torch.optim.AdamW([
        {'params': model.code_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
        {'params': [p for n, p in model.named_parameters() if 'code_encoder' not in n], 'lr': Config.LEARNING_RATE}
    ], weight_decay=0.01)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0
    best_model_state = None

    # ENTRAÎNEMENT
    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels, _) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds) if train_preds else 1
        train_acc = accuracy_score(train_labels_list, train_preds) if train_labels_list else 0
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss if val_loss else 0)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
            print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
            print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
            print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 10:
            print(f"Early stopping déclenché à l'epoch {epoch+1}")
            break

    # Charger le meilleur modèle pour l'évaluation finale
    if best_model_state:
        model.load_state_dict(best_model_state)

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Visualisations
    if train_losses:
        plt.figure(figsize=(15, 5))

        # Courbe 1: Loss
        plt.subplot(1, 3, 1)
        epochs_range = range(1, len(train_losses) + 1)
        plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
        plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
        plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        # Courbe 2: Accuracy
        plt.subplot(1, 3, 2)
        plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
        plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
        plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        # Courbe 3: F1-score et G-mean
        plt.subplot(1, 3, 3)
        plt.plot(epochs_range, f1_scores, 'g-', label='F1-Score')
        plt.plot(epochs_range, gmean_scores, 'm-', label='G-Mean')
        plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean')
        plt.xlabel('Epochs')
        plt.ylabel('Score')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.savefig(f"{Config.SAVE_PATH}training_curves_{model_type}.png", dpi=300, bbox_inches='tight')
        plt.show()

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores,
        'dataset_info': {
            'train_samples': len(X_train),
            'val_samples': len(X_val),
            'features': input_dim,
            'train_distribution': dict(Counter(y_train)),
            'val_distribution': dict(Counter(y_val))
        }
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

# FONCTION PRINCIPALE
def main():
    """Fonction principale pour lancer l'entraînement avec alignement"""
    print("🚀 DÉMARRAGE AVEC ALIGNEMENT EXPLICITE MÉTRIQUES-CODE")
    print("Utilisation de l'attention croisée pour l'alignement multi-modal")
    print("Activation: ReLU")
    print("Gestion corrigée de SMOTE avec réalignement des tokens")

    try:
        # Entraîner les différents modèles avec alignement
        print("\n1. Modèle avec attention croisée bidirectionnelle")
        model_cross, results_cross = train_aligned_model("cross_attention")

        print("\n2. Modèle avec alignement hiérarchique")
        model_hierarchical, results_hierarchical = train_aligned_model("hierarchical")

        # Comparaison des résultats
        print("\n" + "="*60)
        print("COMPARAISON DES MODÈLES AVEC ALIGNEMENT")
        print("="*60)

        print(f"{'Métrique':<20} {'Cross-Attention':<15} {'Hierarchical':<15} {'Meilleur':<15}")
        print("-" * 65)

        metrics = ['F1-Score', 'G-Mean', 'Accuracy', 'Sensitivity', 'Specificity']
        cross_scores = [
            results_cross['best_f1'], results_cross['final_gmean'],
            results_cross['final_accuracy'], results_cross['final_sensitivity'],
            results_cross['final_specificity']
        ]
        hierarchical_scores = [
            results_hierarchical['best_f1'], results_hierarchical['final_gmean'],
            results_hierarchical['final_accuracy'], results_hierarchical['final_sensitivity'],
            results_hierarchical['final_specificity']
        ]

        for metric, cross, hier in zip(metrics, cross_scores, hierarchical_scores):
            better = "Cross" if cross > hier else "Hierarchical" if hier > cross else "Égalité"
            print(f"{metric:<20} {cross:<15.4f} {hier:<15.4f} {better:<15}")

        # Recommandation finale
        best_model = "Cross-Attention" if results_cross['best_f1'] > results_hierarchical['best_f1'] else "Hierarchical"
        print(f"\n✓ Modèle recommandé: {best_model}")
        print(f"✓ Résultats sauvegardés dans: {Config.SAVE_PATH}")

    except Exception as e:
        print(f"❌ Erreur lors de l'entraînement: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter
import re

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    # Nouveaux paramètres pour l'alignement
    MAX_CODE_LENGTH = 512
    CROSS_ATTENTION_HEADS = 8

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé avec alignement - CORRIGÉ
class AlignedBugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels, file_names):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)
        self.file_names = file_names

        # Vérifications de cohérence
        assert len(self.metrics) == len(self.labels) == len(self.file_names), f"Mismatch dans les données alignées: metrics={len(self.metrics)}, labels={len(self.labels)}, filenames={len(self.file_names)}"
        assert len(self.code_tokens['input_ids']) == len(self.labels), f"Mismatch entre code_tokens et labels: tokens={len(self.code_tokens['input_ids'])}, labels={len(self.labels)}"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        # CORRECTION : Utiliser self.code_tokens au lieu de code_tokens
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]  # ← CORRECTION ICI
        }, self.labels[idx], self.file_names[idx]

# Modèle avec alignement par attention croisée avancée
class CrossAttentionAlignedModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()

        # Projection des métriques
        self.metric_projection = nn.Sequential(
            nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder pour les métriques
        self.metrics_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=Config.EMBEDDING_DIM,
                nhead=Config.N_HEADS,
                dropout=Config.DROPOUT,
                activation='relu',
                batch_first=True,
                norm_first=True
            ),
            num_layers=Config.NUM_LAYERS
        )

        # Modèle pré-entraîné pour le code
        self.code_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        self.code_projection = nn.Linear(768, Config.EMBEDDING_DIM)

        # ATTENTION CROISÉE BIDIRECTIONNELLE
        # Métriques → Code
        self.metrics_to_code_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.CROSS_ATTENTION_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Code → Métriques
        self.code_to_metrics_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.CROSS_ATTENTION_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion multi-modale
        self.fusion_layer = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 3, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(Config.EMBEDDING_DIM, Config.EMBEDDING_DIM // 2),
            nn.ReLU()
        )

        # Classifier avec features alignées
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM // 2, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        batch_size = metrics.size(0)

        # Encoder les métriques
        metrics_proj = self.metric_projection(metrics).unsqueeze(1)  # [B, 1, D]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, D]

        # Encoder le code
        code_outputs = self.code_encoder(**code_tokens)
        code_embeddings = self.code_projection(code_outputs.last_hidden_state)  # [B, L, D]

        # ATTENTION CROISÉE BIDIRECTIONNELLE
        # 1. Métriques → Code (les métriques interrogent le code)
        metrics_attended_code, _ = self.metrics_to_code_attention(
            query=metrics_encoded,    # [B, 1, D]
            key=code_embeddings,      # [B, L, D]
            value=code_embeddings     # [B, L, D]
        )  # [B, 1, D]

        # 2. Code → Métriques (le code interroge les métriques)
        # Prendre la moyenne des embeddings de code comme query
        code_mean = code_embeddings.mean(dim=1, keepdim=True)  # [B, 1, D]
        code_attended_metrics, _ = self.code_to_metrics_attention(
            query=code_mean,          # [B, 1, D]
            key=metrics_encoded,      # [B, 1, D]
            value=metrics_encoded     # [B, 1, D]
        )  # [B, 1, D]

        # FUSION MULTI-SOURCE
        combined_features = torch.cat([
            metrics_encoded[:, 0, :],           # Représentation originale des métriques
            metrics_attended_code[:, 0, :],     # Métriques enrichies par le code
            code_attended_metrics[:, 0, :]      # Code enrichi par les métriques
        ], dim=-1)  # [B, 3*D]

        # Fusion finale
        fused = self.fusion_layer(combined_features)  # [B, D/2]

        # Classification
        logits = self.classifier(fused)

        return logits

# [Le reste du code reste identique...]

# FONCTIONS D'ALIGNEMENT AMÉLIORÉES
def extract_filename_from_path(file_path):
    """Extrait le nom de fichier avec plus de flexibilité"""
    if pd.isna(file_path) or file_path is None:
        return []

    base_name = os.path.basename(str(file_path))
    name_without_ext = os.path.splitext(base_name)[0]

    # Nettoyage plus agressif pour trouver des correspondances
    cleaned_name = re.sub(r'[^a-zA-Z0-9]', '', name_without_ext.lower())

    if not cleaned_name:
        return []

    # Essayer plusieurs formats
    variations = set()
    variations.add(cleaned_name)
    variations.add(cleaned_name.replace('test', '').replace('testcase', ''))
    variations.add(re.sub(r'\d+', '', cleaned_name))  # Supprimer les chiffres

    if '_' in cleaned_name:
        variations.add(cleaned_name.split('_')[0])
    if '.' in cleaned_name:
        variations.add(cleaned_name.split('.')[0])

    # Ajouter des variations avec et sans underscores/chiffres
    variations.add(cleaned_name.replace('_', ''))
    variations.add(re.sub(r'[^a-zA-Z]', '', cleaned_name))

    # Filtrer les chaînes vides
    return [v for v in variations if v and len(v) > 1]

def align_metrics_with_code(df_metrics, code_files_dict):
    """Aligne avec des critères plus flexibles"""
    aligned_metrics = []
    aligned_code = []
    aligned_labels = []
    aligned_filenames = []

    matches_count = 0
    partial_matches_count = 0
    no_matches_count = 0

    # Créer un index des fichiers code pour recherche rapide
    code_index = {}
    for code_filename, code_content in code_files_dict.items():
        code_variations = extract_filename_from_path(code_filename)
        for variation in code_variations:
            if variation:  # Éviter les chaînes vides
                code_index[variation] = (code_content, code_filename)

    print(f"✓ Index créé avec {len(code_index)} variations de noms de code")

    for idx, row in df_metrics.iterrows():
        # Extraire le nom de fichier des métriques
        if 'name' in row:
            metric_variations = extract_filename_from_path(str(row['name']))
        else:
            metric_variations = [f"file{idx}"]

        # Chercher le code correspondant
        matching_code = None
        match_type = "none"
        matched_filename = ""

        # Recherche exacte dans toutes les variations
        for metric_var in metric_variations:
            if metric_var in code_index:
                matching_code, matched_filename = code_index[metric_var]
                match_type = "exact"
                matches_count += 1
                break

        # Recherche partielle si pas de correspondance exacte
        if matching_code is None:
            for metric_var in metric_variations:
                for code_var, (code_content, filename) in code_index.items():
                    if (metric_var and code_var and
                        (metric_var in code_var or code_var in metric_var)):
                        # Vérifier que c'est une correspondance significative
                        if len(metric_var) > 3 and len(code_var) > 3:
                            matching_code = code_content
                            matched_filename = filename
                            match_type = "partial"
                            partial_matches_count += 1
                            break
                if matching_code:
                    break

        # Dernier fallback : utiliser le fichier le plus proche
        if matching_code is None and code_files_dict:
            # Essayer de trouver le fichier avec le nom le plus similaire
            best_match = None
            best_score = 0

            for metric_var in metric_variations:
                for code_var, (code_content, filename) in code_index.items():
                    # Calcul de similarité simple
                    score = 0
                    if metric_var in code_var:
                        score = len(metric_var) / len(code_var)
                    elif code_var in metric_var:
                        score = len(code_var) / len(metric_var)

                    if score > best_score:
                        best_score = score
                        best_match = (code_content, filename)

            if best_match and best_score > 0.3:  # Seuil de similarité
                matching_code, matched_filename = best_match
                match_type = "similar"
            else:
                matching_code, matched_filename = next(iter(code_index.values()))
                match_type = "fallback"
            no_matches_count += 1

        if matching_code is not None:
            # Préparer les métriques (exclure 'name' et 'bug')
            metric_data = {}
            for col, value in row.items():
                if col not in ['name', 'bug'] and pd.api.types.is_numeric_dtype(type(value)):
                    metric_data[col] = float(value) if pd.notna(value) else 0.0

            aligned_metrics.append(metric_data)
            aligned_code.append(matching_code)
            aligned_labels.append(1 if row.get('bug', 0) > 0 else 0)
            aligned_filenames.append(f"{matched_filename}_{match_type}")

    print(f"✓ Alignement: {matches_count} exacts, {partial_matches_count} partiels, {no_matches_count} fallbacks/similaires")
    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

def load_and_align_data(metrics_path, code_path):
    """Charge et aligne les métriques avec le code source"""
    # Charger les métriques
    df_metrics = pd.read_csv(metrics_path)
    print(f"✓ {len(df_metrics)} métriques chargées depuis {os.path.basename(metrics_path)}")

    # Charger tous les fichiers code dans un dictionnaire
    code_files_dict = {}
    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        file_path = os.path.join(root, file)
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            code_content = f.read().strip()
                            if code_content and len(code_content) > 10:  # Fichier non vide
                                code_files_dict[file] = code_content
                    except Exception as e:
                        print(f"Erreur lecture {file}: {e}")
                        continue

    print(f"✓ {len(code_files_dict)} fichiers code chargés depuis {os.path.basename(code_path)}")

    if not code_files_dict:
        print("⚠️ Aucun fichier code trouvé!")
        return [], [], [], []

    # Aligner les données
    aligned_metrics, aligned_code, aligned_labels, aligned_filenames = align_metrics_with_code(
        df_metrics, code_files_dict
    )

    print(f"✓ {len(aligned_metrics)} échantillons alignés sur {len(df_metrics)} métriques originales")

    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

# FONCTIONS DE PRÉTRAITEMENT AVEC ALIGNEMENT
def preprocess_aligned_metrics(metrics_list, labels):
    """Prétraite les métriques alignées AVEC les labels"""
    if not metrics_list:
        return np.array([]), np.array([]), None

    df_metrics = pd.DataFrame(metrics_list)

    # Gérer les valeurs manquantes
    if df_metrics.isnull().any().any():
        df_metrics = df_metrics.fillna(df_metrics.median())

    # Supprimer les features à variance nulle
    var_threshold = 0.01
    variances = df_metrics.var()
    low_var_cols = variances < var_threshold
    if low_var_cols.any():
        df_metrics = df_metrics.loc[:, ~low_var_cols]
        print(f"✓ {low_var_cols.sum()} colonnes à faible variance supprimées")

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(df_metrics)

    print(f"✓ Métriques prétraitées: {features_scaled.shape}")
    return features_scaled, np.array(labels), scaler

# FONCTIONS UTILITAIRES CORRIGÉES
def apply_smote_with_realignment(X, y, code_tokens, filenames):
    """Applique SMOTE et réaligne correctement les tokens de code ET les filenames"""
    print(f"Distribution avant SMOTE: {Counter(y)}")

    original_size = len(X)

    # Appliquer SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)

    new_size = len(X_resampled)
    print(f"Distribution après SMOTE: {Counter(y_resampled)}")

    # Réaligner les tokens de code ET les filenames
    if new_size <= original_size:
        # Sous-échantillonnage : prendre les premiers
        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][:new_size],
            'attention_mask': code_tokens['attention_mask'][:new_size]
        }
        filenames_resampled = filenames[:new_size]
    else:
        # Sur-échantillonnage : dupliquer aléatoirement
        indices = list(range(original_size)) + \
                 list(np.random.randint(0, original_size, new_size - original_size))

        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][indices],
            'attention_mask': code_tokens['attention_mask'][indices]
        }
        filenames_resampled = [filenames[i] for i in indices]

    print(f"✓ Tokens et filenames réalignés: {original_size} → {new_size} échantillons")
    return X_resampled, y_resampled, code_tokens_resampled, filenames_resampled

def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    print(f"✓ Poids des classes: {weights}")
    return weight_tensor

def prepare_aligned_dataset_with_smote(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset avec alignement explicite et gestion correcte de SMOTE"""
    # Charger et aligner les données
    metrics_list, code_lines, labels, filenames = load_and_align_data(metrics_path, code_path)

    if not metrics_list:
        raise ValueError("Aucune donnée alignée trouvée")

    # Prétraiter les métriques
    features, labels, scaler = preprocess_aligned_metrics(metrics_list, labels)

    # Tokeniser le code AVANT SMOTE
    original_code_tokens = tokenizer(
        code_lines,
        padding=True,
        truncation=True,
        return_tensors='pt',
        max_length=Config.MAX_CODE_LENGTH
    )

    # Équilibrer les classes SI demandé avec réalignement correct
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_tokens, filenames = apply_smote_with_realignment(
                features, labels, original_code_tokens, filenames
            )
            print("✓ SMOTE appliqué avec réalignement correct des tokens et filenames")
            return features, code_tokens, labels, scaler, filenames

        except Exception as e:
            print(f"Erreur équilibrage: {e}, utilisation des données originales")

    return features, original_code_tokens, labels, scaler, filenames

# FONCTIONS D'ÉVALUATION
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels, _ in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# FONCTION D'ENTRAÎNEMENT CORRIGÉE
def train_aligned_model(model_type="cross_attention"):
    """Entraîne un modèle avec alignement explicite - VERSION CORRIGÉE"""
    print(f"\n{'='*60}")
    print(f"ENTRAÎNEMENT AVEC ALIGNEMENT: {model_type.upper()}")
    print(f"{'='*60}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Chemins des données
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/xerces/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/xerces/"

    train_versions = ["1.1", "1.2", "1.3"]
    val_version = "1.4.4"

    # Initialisation correcte des listes
    all_train_features = []
    all_train_labels = []
    all_train_filenames = []
    all_train_code_tokens = {'input_ids': [], 'attention_mask': []}

    # Charger les données d'entraînement AVEC SMOTE INDIVIDUEL
    for version in train_versions:
        metrics_path = f"{base_metrics_path}xerces-{version}.csv"
        code_path = f"{base_code_path}xerces-{version}"

        try:
            # Appliquer SMOTE individuellement sur chaque version
            features, code_tokens, labels, _, filenames = prepare_aligned_dataset_with_smote(
                metrics_path, code_path, tokenizer, apply_balancing=True
            )

            all_train_features.append(features)
            all_train_labels.extend(labels)
            all_train_filenames.extend(filenames)

            all_train_code_tokens['input_ids'].append(code_tokens['input_ids'])
            all_train_code_tokens['attention_mask'].append(code_tokens['attention_mask'])

            print(f"✓ Version {version}: {len(features)} échantillons alignés et équilibrés")

        except Exception as e:
            print(f"✗ Erreur avec la version {version}: {e}")
            continue

    # Vérifier qu'on a des données
    if not all_train_features:
        raise ValueError("Aucune donnée d'entraînement chargée")

    # Concaténer toutes les données d'entraînement
    X_train = np.vstack(all_train_features)
    y_train = np.array(all_train_labels)

    # Concaténer les tokens
    all_train_code = {
        'input_ids': torch.cat(all_train_code_tokens['input_ids']),
        'attention_mask': torch.cat(all_train_code_tokens['attention_mask'])
    }

    print(f"✓ Données d'entraînement finales: {X_train.shape[0]} échantillons")
    print(f"✓ Distribution: {Counter(y_train)}")

    # Ajustement automatique si mismatch
    min_length = min(len(X_train), len(y_train), len(all_train_filenames), all_train_code['input_ids'].shape[0])
    if min_length < len(X_train):
        X_train = X_train[:min_length]
        y_train = y_train[:min_length]
        all_train_filenames = all_train_filenames[:min_length]
        all_train_code = {
            'input_ids': all_train_code['input_ids'][:min_length],
            'attention_mask': all_train_code['attention_mask'][:min_length]
        }
        print(f"✓ Données ajustées à {min_length} échantillons")

    # Charger les données de validation
    print("\nChargement des données de validation...")
    metrics_val_path = f"{base_metrics_path}xerces-{val_version}.csv"
    code_val_path = f"{base_code_path}xerces-{val_version}"

    X_val, code_val_tokens, y_val, _, val_filenames = prepare_aligned_dataset_with_smote(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    print(f"✓ Données de validation: {X_val.shape[0]} échantillons")

    # Ajustement automatique de la validation
    min_length_val = min(len(X_val), len(y_val), len(val_filenames), code_val_tokens['input_ids'].shape[0])
    if min_length_val < len(X_val):
        X_val = X_val[:min_length_val]
        y_val = y_val[:min_length_val]
        val_filenames = val_filenames[:min_length_val]
        code_val_tokens = {
            'input_ids': code_val_tokens['input_ids'][:min_length_val],
            'attention_mask': code_val_tokens['attention_mask'][:min_length_val]
        }
        print(f"✓ Validation ajustée à {min_length_val} échantillons")

    # Vérifier la cohérence des features
    assert X_train.shape[1] == X_val.shape[1], f"Mismatch features: train{X_train.shape[1]} vs val{X_val.shape[1]}"

    # Créer les datasets
    train_ds = AlignedBugDataset(X_train, all_train_code, y_train, all_train_filenames)
    val_ds = AlignedBugDataset(X_val, code_val_tokens, y_val, val_filenames)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Initialiser le modèle avec la dimension exacte
    input_dim = X_train.shape[1]
    if model_type == "hierarchical":
        model = HierarchicalAlignedModel(input_metric_dim=input_dim).to(Config.DEVICE)
    else:
        model = CrossAttentionAlignedModel(input_metric_dim=input_dim).to(Config.DEVICE)

    print(f"✓ Modèle {model_type} initialisé avec {input_dim} features")

    # Optimiseur et loss
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    optimizer = torch.optim.AdamW([
        {'params': model.code_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
        {'params': [p for n, p in model.named_parameters() if 'code_encoder' not in n], 'lr': Config.LEARNING_RATE}
    ], weight_decay=0.01)

    # CORRECTION : Scheduler sans le paramètre 'verbose'
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0
    best_model_state = None

    # ENTRAÎNEMENT
    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels, _) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds) if train_preds else 1
        train_acc = accuracy_score(train_labels_list, train_preds) if train_labels_list else 0
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss if val_loss else 0)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
            print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
            print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
            print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 10:
            print(f"Early stopping déclenché à l'epoch {epoch+1}")
            break

    # Charger le meilleur modèle pour l'évaluation finale
    if best_model_state:
        model.load_state_dict(best_model_state)

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Visualisations
    if train_losses:
        plt.figure(figsize=(15, 5))

        # Courbe 1: Loss
        plt.subplot(1, 3, 1)
        epochs_range = range(1, len(train_losses) + 1)
        plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
        plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
        plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        # Courbe 2: Accuracy
        plt.subplot(1, 3, 2)
        plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
        plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
        plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        # Courbe 3: F1-score et G-mean
        plt.subplot(1, 3, 3)
        plt.plot(epochs_range, f1_scores, 'g-', label='F1-Score')
        plt.plot(epochs_range, gmean_scores, 'm-', label='G-Mean')
        plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean')
        plt.xlabel('Epochs')
        plt.ylabel('Score')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.savefig(f"{Config.SAVE_PATH}training_curves_{model_type}.png", dpi=300, bbox_inches='tight')
        plt.show()

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

def main():
    """Fonction principale pour lancer l'entraînement avec alignement"""
    print("🚀 DÉMARRAGE AVEC ALIGNEMENT EXPLICITE MÉTRIQUES-CODE")
    print("Utilisation de l'attention croisée pour l'alignement multi-modal")
    print("Activation: ReLU")
    print("Gestion corrigée de SMOTE avec réalignement des tokens")

    try:
        # Entraîner les différents modèles avec alignement
        print("\n1. Modèle avec attention croisée bidirectionnelle")
        model_cross, results_cross = train_aligned_model("cross_attention")

        print("\n2. Modèle avec alignement hiérarchique")
        model_hierarchical, results_hierarchical = train_aligned_model("hierarchical")

        # Comparaison des résultats
        print("\n" + "="*60)
        print("COMPARAISON DES MODÈLES AVEC ALIGNEMENT")
        print("="*60)

        print(f"{'Métrique':<20} {'Cross-Attention':<15} {'Hierarchical':<15} {'Meilleur':<15}")
        print("-" * 65)

        metrics = ['F1-Score', 'G-Mean', 'Accuracy', 'Sensitivity', 'Specificity']
        cross_scores = [
            results_cross['best_f1'], results_cross['final_gmean'],
            results_cross['final_accuracy'], results_cross['final_sensitivity'],
            results_cross['final_specificity']
        ]
        hierarchical_scores = [
            results_hierarchical['best_f1'], results_hierarchical['final_gmean'],
            results_hierarchical['final_accuracy'], results_hierarchical['final_sensitivity'],
            results_hierarchical['final_specificity']
        ]

        for metric, cross, hier in zip(metrics, cross_scores, hierarchical_scores):
            better = "Cross" if cross > hier else "Hierarchical" if hier > cross else "Égalité"
            print(f"{metric:<20} {cross:<15.4f} {hier:<15.4f} {better:<15}")

        # Recommandation finale
        best_model = "Cross-Attention" if results_cross['best_f1'] > results_hierarchical['best_f1'] else "Hierarchical"
        print(f"\n✓ Modèle recommandé: {best_model}")
        print(f"✓ Résultats sauvegardés dans: {Config.SAVE_PATH}")

    except Exception as e:
        print(f"❌ Erreur lors de l'entraînement: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter
import re

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    # Paramètres pour l'alignement
    MAX_CODE_LENGTH = 512

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé avec alignement
class AlignedBugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels, file_names):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)
        self.file_names = file_names

        # Vérifications de cohérence
        assert len(self.metrics) == len(self.labels) == len(self.file_names), f"Mismatch dans les données alignées: metrics={len(self.metrics)}, labels={len(self.labels)}, filenames={len(self.file_names)}"
        assert len(self.code_tokens['input_ids']) == len(self.labels), f"Mismatch entre code_tokens et labels: tokens={len(self.code_tokens['input_ids'])}, labels={len(self.labels)}"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx], self.file_names[idx]

# STRATÉGIE 1: Modèle hybride (encoder from scratch + decoder pré-entraîné)
class HybridTransBugModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()

        print(f"🔧 Initialisation du modèle HYBRIDE - Encoder from scratch + Decoder pré-entraîné")
        print(f"   - Dimension métriques: {input_metric_dim} -> {Config.EMBEDDING_DIM}")
        print(f"   - Encoder: Transformer from scratch ({Config.NUM_LAYERS} couches)")
        print(f"   - Decoder: RoBERTa pré-entraîné")

        # Projection des métriques vers la dimension d'embedding
        self.metric_projection = nn.Sequential(
            nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder transformer FROM SCRATCH pour les métriques
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='relu',
            batch_first=True,
            norm_first=True
        )
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder = RobertaModel.from_pretrained("microsoft/codebert-base")

        # Projection pour aligner les dimensions RoBERTa -> Config.EMBEDDING_DIM
        self.code_projection = nn.Linear(768, Config.EMBEDDING_DIM)

        # Couche d'attention pour combiner les représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec encoder from scratch
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, D]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, D]

        # Traitement du code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, D]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query (métriques)
            code_embed,       # Key (code)
            code_embed        # Value (code)
        )

        # Fusion avec concaténation
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# STRATÉGIE 2: Modèle entièrement pré-entraîné (encoder et decoder pré-entraînés)
class FullPretrainedModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()

        print(f"🔧 Initialisation du modèle FULL PRETRAINED - Encoder et Decoder pré-entraînés")
        print(f"   - Dimension métriques: {input_metric_dim} -> {Config.EMBEDDING_DIM}")
        print(f"   - Encoder: RoBERTa pré-entraîné")
        print(f"   - Decoder: RoBERTa pré-entraîné")

        # Encoder PRÉ-ENTRAÎNÉ pour les métriques (adaptation de RoBERTa)
        self.metrics_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")

        # Projection pour les métriques numériques vers le format attendu par transformer
        self.metrics_projection = nn.Sequential(
            nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder = RobertaModel.from_pretrained("microsoft/codebert-base")

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(768, Config.EMBEDDING_DIM)

        # Couche d'attention croisée pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, D]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, D]

        # Encoder le code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_encoded = self.code_projection(code_encoded)  # [B, seq_len, D]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query (métriques)
            code_encoded,     # Key (code)
            code_encoded      # Value (code)
        )

        # Fusion avec concaténation
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# FONCTIONS D'ALIGNEMENT
def extract_filename_from_path(file_path):
    """Extrait le nom de fichier avec plus de flexibilité"""
    if pd.isna(file_path) or file_path is None:
        return []

    base_name = os.path.basename(str(file_path))
    name_without_ext = os.path.splitext(base_name)[0]

    # Nettoyage pour trouver des correspondances
    cleaned_name = re.sub(r'[^a-zA-Z0-9]', '', name_without_ext.lower())

    if not cleaned_name:
        return []

    # Essayer plusieurs formats
    variations = set()
    variations.add(cleaned_name)
    variations.add(cleaned_name.replace('test', '').replace('testcase', ''))
    variations.add(re.sub(r'\d+', '', cleaned_name))

    if '_' in cleaned_name:
        variations.add(cleaned_name.split('_')[0])
    if '.' in cleaned_name:
        variations.add(cleaned_name.split('.')[0])

    variations.add(cleaned_name.replace('_', ''))
    variations.add(re.sub(r'[^a-zA-Z]', '', cleaned_name))

    return [v for v in variations if v and len(v) > 1]

def align_metrics_with_code(df_metrics, code_files_dict):
    """Aligne explicitement les métriques avec le code source correspondant"""
    aligned_metrics = []
    aligned_code = []
    aligned_labels = []
    aligned_filenames = []

    matches_count = 0
    partial_matches_count = 0
    no_matches_count = 0

    # Créer un index des fichiers code pour recherche rapide
    code_index = {}
    for code_filename, code_content in code_files_dict.items():
        code_variations = extract_filename_from_path(code_filename)
        for variation in code_variations:
            if variation:
                code_index[variation] = (code_content, code_filename)

    print(f"✓ Index créé avec {len(code_index)} variations de noms de code")

    for idx, row in df_metrics.iterrows():
        # Extraire le nom de fichier des métriques
        if 'name' in row:
            metric_variations = extract_filename_from_path(str(row['name']))
        else:
            metric_variations = [f"file{idx}"]

        # Chercher le code correspondant
        matching_code = None
        match_type = "none"
        matched_filename = ""

        # Recherche exacte
        for metric_var in metric_variations:
            if metric_var in code_index:
                matching_code, matched_filename = code_index[metric_var]
                match_type = "exact"
                matches_count += 1
                break

        # Recherche partielle
        if matching_code is None:
            for metric_var in metric_variations:
                for code_var, (code_content, filename) in code_index.items():
                    if (metric_var and code_var and
                        (metric_var in code_var or code_var in metric_var)):
                        if len(metric_var) > 3 and len(code_var) > 3:
                            matching_code = code_content
                            matched_filename = filename
                            match_type = "partial"
                            partial_matches_count += 1
                            break
                if matching_code:
                    break

        # Fallback
        if matching_code is None and code_files_dict:
            matching_code, matched_filename = next(iter(code_index.values()))
            match_type = "fallback"
            no_matches_count += 1

        if matching_code is not None:
            # Préparer les métriques
            metric_data = {}
            for col, value in row.items():
                if col not in ['name', 'bug'] and pd.api.types.is_numeric_dtype(type(value)):
                    metric_data[col] = float(value) if pd.notna(value) else 0.0

            aligned_metrics.append(metric_data)
            aligned_code.append(matching_code)
            aligned_labels.append(1 if row.get('bug', 0) > 0 else 0)
            aligned_filenames.append(f"{matched_filename}_{match_type}")

    print(f"✓ Alignement: {matches_count} exacts, {partial_matches_count} partiels, {no_matches_count} fallbacks")
    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

def load_and_align_data(metrics_path, code_path):
    """Charge et aligne les métriques avec le code source"""
    # Charger les métriques
    df_metrics = pd.read_csv(metrics_path)
    print(f"✓ {len(df_metrics)} métriques chargées depuis {os.path.basename(metrics_path)}")

    # Charger tous les fichiers code
    code_files_dict = {}
    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        file_path = os.path.join(root, file)
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            code_content = f.read().strip()
                            if code_content and len(code_content) > 10:
                                code_files_dict[file] = code_content
                    except Exception as e:
                        continue

    print(f"✓ {len(code_files_dict)} fichiers code chargés depuis {os.path.basename(code_path)}")

    if not code_files_dict:
        print("⚠️ Aucun fichier code trouvé!")
        return [], [], [], []

    # Aligner les données
    aligned_metrics, aligned_code, aligned_labels, aligned_filenames = align_metrics_with_code(
        df_metrics, code_files_dict
    )

    print(f"✓ {len(aligned_metrics)} échantillons alignés sur {len(df_metrics)} métriques originales")

    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

# FONCTIONS DE PRÉTRAITEMENT
def preprocess_aligned_metrics(metrics_list, labels):
    """Prétraite les métriques alignées"""
    if not metrics_list:
        return np.array([]), np.array([]), None

    df_metrics = pd.DataFrame(metrics_list)

    # Gérer les valeurs manquantes
    if df_metrics.isnull().any().any():
        df_metrics = df_metrics.fillna(df_metrics.median())

    # Supprimer les features à variance nulle
    var_threshold = 0.01
    variances = df_metrics.var()
    low_var_cols = variances < var_threshold
    if low_var_cols.any():
        df_metrics = df_metrics.loc[:, ~low_var_cols]
        print(f"✓ {low_var_cols.sum()} colonnes à faible variance supprimées")

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(df_metrics)

    print(f"✓ Métriques prétraitées: {features_scaled.shape}")
    return features_scaled, np.array(labels), scaler

def apply_smote_with_realignment(X, y, code_tokens, filenames):
    """Applique SMOTE et réaligne correctement les tokens de code ET les filenames"""
    print(f"Distribution avant SMOTE: {Counter(y)}")

    original_size = len(X)

    # Appliquer SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)

    new_size = len(X_resampled)
    print(f"Distribution après SMOTE: {Counter(y_resampled)}")

    # Réaligner les tokens de code ET les filenames
    if new_size <= original_size:
        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][:new_size],
            'attention_mask': code_tokens['attention_mask'][:new_size]
        }
        filenames_resampled = filenames[:new_size]
    else:
        indices = list(range(original_size)) + \
                 list(np.random.randint(0, original_size, new_size - original_size))

        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][indices],
            'attention_mask': code_tokens['attention_mask'][indices]
        }
        filenames_resampled = [filenames[i] for i in indices]

    print(f"✓ Tokens et filenames réalignés: {original_size} → {new_size} échantillons")
    return X_resampled, y_resampled, code_tokens_resampled, filenames_resampled

def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    print(f"✓ Poids des classes: {weights}")
    return weight_tensor

def prepare_aligned_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset avec alignement explicite"""
    # Charger et aligner les données
    metrics_list, code_lines, labels, filenames = load_and_align_data(metrics_path, code_path)

    if not metrics_list:
        raise ValueError("Aucune donnée alignée trouvée")

    # Prétraiter les métriques
    features, labels, scaler = preprocess_aligned_metrics(metrics_list, labels)

    # Tokeniser le code AVANT SMOTE
    original_code_tokens = tokenizer(
        code_lines,
        padding=True,
        truncation=True,
        return_tensors='pt',
        max_length=Config.MAX_CODE_LENGTH
    )

    # Équilibrer les classes SI demandé avec réalignement correct
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_tokens, filenames = apply_smote_with_realignment(
                features, labels, original_code_tokens, filenames
            )
            print("✓ SMOTE appliqué avec réalignement correct")
            return features, code_tokens, labels, scaler, filenames

        except Exception as e:
            print(f"Erreur équilibrage: {e}, utilisation des données originales")

    return features, original_code_tokens, labels, scaler, filenames

# FONCTIONS D'ÉVALUATION
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels, _ in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# FONCTION D'ENTRAÎNEMENT
def train_model(model_type="hybrid"):
    """Entraîne un modèle avec alignement explicite"""
    print(f"\n{'='*60}")
    print(f"ENTRAÎNEMENT: {model_type.upper()}")
    print(f"{'='*60}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Chemins des données
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/xerces/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/xerces/"

    train_versions = ["1.1", "1.2", "1.3"]
    val_version = "1.4.4"

    # Charger les données d'entraînement
    all_train_features = []
    all_train_labels = []
    all_train_filenames = []
    all_train_code_tokens = {'input_ids': [], 'attention_mask': []}

    for version in train_versions:
        metrics_path = f"{base_metrics_path}xerces-{version}.csv"
        code_path = f"{base_code_path}xerces-{version}"

        try:
            features, code_tokens, labels, _, filenames = prepare_aligned_dataset(
                metrics_path, code_path, tokenizer, apply_balancing=True
            )

            all_train_features.append(features)
            all_train_labels.extend(labels)
            all_train_filenames.extend(filenames)

            all_train_code_tokens['input_ids'].append(code_tokens['input_ids'])
            all_train_code_tokens['attention_mask'].append(code_tokens['attention_mask'])

            print(f"✓ Version {version}: {len(features)} échantillons alignés")

        except Exception as e:
            print(f"✗ Erreur avec la version {version}: {e}")
            continue

    if not all_train_features:
        raise ValueError("Aucune donnée d'entraînement chargée")

    # Concaténer toutes les données d'entraînement
    X_train = np.vstack(all_train_features)
    y_train = np.array(all_train_labels)

    # Concaténer les tokens
    all_train_code = {
        'input_ids': torch.cat(all_train_code_tokens['input_ids']),
        'attention_mask': torch.cat(all_train_code_tokens['attention_mask'])
    }

    print(f"✓ Données d'entraînement finales: {X_train.shape[0]} échantillons")
    print(f"✓ Distribution: {Counter(y_train)}")

    # Ajustement automatique si mismatch
    min_length = min(len(X_train), len(y_train), len(all_train_filenames), all_train_code['input_ids'].shape[0])
    if min_length < len(X_train):
        X_train = X_train[:min_length]
        y_train = y_train[:min_length]
        all_train_filenames = all_train_filenames[:min_length]
        all_train_code = {
            'input_ids': all_train_code['input_ids'][:min_length],
            'attention_mask': all_train_code['attention_mask'][:min_length]
        }
        print(f"✓ Données ajustées à {min_length} échantillons")

    # Charger les données de validation
    print("\nChargement des données de validation...")
    metrics_val_path = f"{base_metrics_path}xerces-{val_version}.csv"
    code_val_path = f"{base_code_path}xerces-{val_version}"

    X_val, code_val_tokens, y_val, _, val_filenames = prepare_aligned_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    print(f"✓ Données de validation: {X_val.shape[0]} échantillons")

    # Ajustement automatique de la validation
    min_length_val = min(len(X_val), len(y_val), len(val_filenames), code_val_tokens['input_ids'].shape[0])
    if min_length_val < len(X_val):
        X_val = X_val[:min_length_val]
        y_val = y_val[:min_length_val]
        val_filenames = val_filenames[:min_length_val]
        code_val_tokens = {
            'input_ids': code_val_tokens['input_ids'][:min_length_val],
            'attention_mask': code_val_tokens['attention_mask'][:min_length_val]
        }
        print(f"✓ Validation ajustée à {min_length_val} échantillons")

    # Vérifier la cohérence des features
    assert X_train.shape[1] == X_val.shape[1], f"Mismatch features: train{X_train.shape[1]} vs val{X_val.shape[1]}"

    # Créer les datasets
    train_ds = AlignedBugDataset(X_train, all_train_code, y_train, all_train_filenames)
    val_ds = AlignedBugDataset(X_val, code_val_tokens, y_val, val_filenames)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Initialiser le modèle selon la stratégie
    input_dim = X_train.shape[1]
    if model_type == "full_pretrained":
        model = FullPretrainedModel(input_metric_dim=input_dim).to(Config.DEVICE)
        model_name = "Full Pretrained"
    else:
        model = HybridTransBugModel(input_metric_dim=input_dim).to(Config.DEVICE)
        model_name = "Hybrid"

    print(f"✓ Modèle {model_name} initialisé avec {input_dim} features")

    # Optimiseur avec learning rates différenciés
    if model_type == "full_pretrained":
        # Pour le modèle entièrement pré-entraîné, utiliser des LR plus faibles
        optimizer = torch.optim.AdamW([
            {'params': model.metrics_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters()
                       if 'metrics_encoder' not in n and 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)
    else:
        # Pour le modèle hybride, seul le decoder est pré-entraîné
        optimizer = torch.optim.AdamW([
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    # Loss avec pondération
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0
    best_model_state = None

    # ENTRAÎNEMENT
    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels, _) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds) if train_preds else 1
        train_acc = accuracy_score(train_labels_list, train_preds) if train_labels_list else 0
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss if val_loss else 0)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
            print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
            print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 10:
            print(f"Early stopping déclenché à l'epoch {epoch+1}")
            break

    # Charger le meilleur modèle
    if best_model_state:
        model.load_state_dict(best_model_state)

    print(f"\n{model_name.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'model_name': model_name,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

# FONCTION PRINCIPALE DE COMPARAISON
def main():
    """Fonction principale pour comparer les deux stratégies"""
    print("🚀 COMPARAISON DES STRATÉGIES TRANSFORMERS POUR LA DÉTECTION DE BUGS")
    print("=" * 80)
    print("STRATÉGIE 1: HYBRIDE - Encoder from scratch + Decoder pré-entraîné")
    print("              • Métriques → Encoder transformer from scratch")
    print("              • Code source → Decoder RoBERTa pré-entraîné")
    print("              • Attention croisée entre les deux représentations")
    print()
    print("STRATÉGIE 2: FULL PRETRAINED - Encoder et Decoder pré-entraînés")
    print("              • Métriques → Encoder RoBERTa pré-entraîné")
    print("              • Code source → Decoder RoBERTa pré-entraîné")
    print("              • Attention croisée entre les deux représentations")
    print("=" * 80)
    print("OBJECTIF: Évaluer l'apport du code source et de la puissance des transformers")
    print("=" * 80)

    results = {}

    try:
        # Entraîner le modèle hybride
        print("\n" + "="*50)
        print("1. STRATÉGIE HYBRIDE (Encoder from scratch + Decoder pré-entraîné)")
        print("="*50)
        model_hybrid, results_hybrid = train_model("hybrid")
        results['hybrid'] = results_hybrid

        # Entraîner le modèle entièrement pré-entraîné
        print("\n" + "="*50)
        print("2. STRATÉGIE FULL PRETRAINED (Encoder et Decoder pré-entraînés)")
        print("="*50)
        model_full, results_full = train_model("full_pretrained")
        results['full_pretrained'] = results_full

        # COMPARAISON DÉTAILLÉE
        print("\n" + "="*80)
        print("COMPARAISON FINALE DES STRATÉGIES")
        print("="*80)

        print(f"{'Métrique':<20} {'Hybride':<12} {'Full Pretrained':<16} {'Différence':<12} {'Meilleur':<10}")
        print("-" * 80)

        metrics = [
            ('F1-Score', results_hybrid['best_f1'], results_full['best_f1']),
            ('G-Mean', results_hybrid['final_gmean'], results_full['final_gmean']),
            ('Accuracy', results_hybrid['final_accuracy'], results_full['final_accuracy']),
            ('Sensitivity', results_hybrid['final_sensitivity'], results_full['final_sensitivity']),
            ('Specificity', results_hybrid['final_specificity'], results_full['final_specificity'])
        ]

        for metric_name, hybrid_score, full_score in metrics:
            difference = hybrid_score - full_score
            better = "Hybride" if hybrid_score > full_score else "Full Pretrained" if full_score > hybrid_score else "Égalité"
            diff_sign = "+" if difference > 0 else ""
            print(f"{metric_name:<20} {hybrid_score:<12.4f} {full_score:<16.4f} {diff_sign}{difference:<11.4f} {better:<10}")

        # ANALYSE DES RÉSULTATS
        print("\n" + "="*80)
        print("ANALYSE DES RÉSULTATS")
        print("="*80)

        if results_hybrid['best_f1'] > results_full['best_f1']:
            best_model = "HYBRIDE"
            best_score = results_hybrid['best_f1']
            advantage = f"+{(results_hybrid['best_f1'] - results_full['best_f1']):.4f}"
            print(f"🎯 STRATÉGIE GAGNANTE: {best_model}")
            print(f"📊 Avantage F1-Score: {advantage}")
            print("💡 INTERPRÉTATION: L'encoder from scratch pour les métriques combiné")
            print("   avec le decoder pré-entraîné pour le code source donne de meilleurs résultats.")
            print("   Cela suggère que l'adaptation spécifique aux métriques est cruciale.")

        else:
            best_model = "FULL PRETRAINED"
            best_score = results_full['best_f1']
            advantage = f"+{(results_full['best_f1'] - results_hybrid['best_f1']):.4f}"
            print(f"🎯 STRATÉGIE GAGNANTE: {best_model}")
            print(f"📊 Avantage F1-Score: {advantage}")
            print("💡 INTERPRÉTATION: La connaissance pré-entraînée pour les deux modalités")
            print("   (métriques et code) donne de meilleurs résultats.")
            print("   Cela montre la puissance des représentations pré-entraînées.")

        print(f"\n✅ Résultats sauvegardés dans: {Config.SAVE_PATH}")

    except Exception as e:
        print(f"❌ Erreur lors de la comparaison: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter
import re

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    # Paramètres pour l'alignement
    MAX_CODE_LENGTH = 512

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé avec alignement
class AlignedBugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels, file_names):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)
        self.file_names = file_names

        # Vérifications de cohérence
        assert len(self.metrics) == len(self.labels) == len(self.file_names), f"Mismatch dans les données alignées: metrics={len(self.metrics)}, labels={len(self.labels)}, filenames={len(self.file_names)}"
        assert len(self.code_tokens['input_ids']) == len(self.labels), f"Mismatch entre code_tokens et labels: tokens={len(self.code_tokens['input_ids'])}, labels={len(self.labels)}"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx], self.file_names[idx]

# STRATÉGIE 1: Modèle hybride (encoder from scratch + decoder pré-entraîné)
class HybridTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()

        print(f"🔧 Initialisation du modèle HYBRIDE - Encoder from scratch + Decoder pré-entraîné")
        print(f"   - Dimension métriques: flexible -> {Config.EMBEDDING_DIM} (LazyLinear)")
        print(f"   - Encoder: Transformer from scratch ({Config.NUM_LAYERS} couches)")
        print(f"   - Decoder: RoBERTa pré-entraîné")

        # Projection des métriques vers la dimension d'embedding AVEC LazyLinear
        self.metric_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),  # ← LazyLinear pour flexibilité
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder transformer FROM SCRATCH pour les métriques
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='relu',
            batch_first=True,
            norm_first=True
        )
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder = RobertaModel.from_pretrained("microsoft/codebert-base")

        # Projection pour aligner les dimensions RoBERTa -> Config.EMBEDDING_DIM
        self.code_projection = nn.Linear(768, Config.EMBEDDING_DIM)

        # Couche d'attention pour combiner les représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations AVEC LazyLinear
        self.fusion = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),  # ← LazyLinear
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final AVEC LazyLinear
        self.classifier = nn.Sequential(
            nn.LazyLinear(256),  # ← LazyLinear
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.LazyLinear(128),  # ← LazyLinear
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.LazyLinear(Config.NUM_CLASSES)  # ← LazyLinear
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec encoder from scratch
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, D]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, D]

        # Traitement du code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, D]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query (métriques)
            code_embed,       # Key (code)
            code_embed        # Value (code)
        )

        # Fusion avec concaténation
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# STRATÉGIE 2: Modèle entièrement pré-entraîné (encoder et decoder pré-entraînés)
class FullPretrainedModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()

        print(f"🔧 Initialisation du modèle FULL PRETRAINED - Encoder et Decoder pré-entraînés")
        print(f"   - Dimension métriques: flexible -> {Config.EMBEDDING_DIM} (LazyLinear)")
        print(f"   - Encoder: RoBERTa pré-entraîné")
        print(f"   - Decoder: RoBERTa pré-entraîné")

        # Encoder PRÉ-ENTRAÎNÉ pour les métriques (adaptation de RoBERTa)
        self.metrics_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")

        # Projection pour les métriques numériques AVEC LazyLinear
        self.metrics_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),  # ← LazyLinear pour flexibilité
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder = RobertaModel.from_pretrained("microsoft/codebert-base")

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(768, Config.EMBEDDING_DIM)

        # Couche d'attention croisée pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations AVEC LazyLinear
        self.fusion = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),  # ← LazyLinear
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final AVEC LazyLinear
        self.classifier = nn.Sequential(
            nn.LazyLinear(256),  # ← LazyLinear
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.LazyLinear(128),  # ← LazyLinear
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.LazyLinear(Config.NUM_CLASSES)  # ← LazyLinear
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, D]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, D]

        # Encoder le code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_encoded = self.code_projection(code_encoded)  # [B, seq_len, D]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query (métriques)
            code_encoded,     # Key (code)
            code_encoded      # Value (code)
        )

        # Fusion avec concaténation
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# FONCTIONS D'ALIGNEMENT
def extract_filename_from_path(file_path):
    """Extrait le nom de fichier avec plus de flexibilité"""
    if pd.isna(file_path) or file_path is None:
        return []

    base_name = os.path.basename(str(file_path))
    name_without_ext = os.path.splitext(base_name)[0]

    # Nettoyage pour trouver des correspondances
    cleaned_name = re.sub(r'[^a-zA-Z0-9]', '', name_without_ext.lower())

    if not cleaned_name:
        return []

    # Essayer plusieurs formats
    variations = set()
    variations.add(cleaned_name)
    variations.add(cleaned_name.replace('test', '').replace('testcase', ''))
    variations.add(re.sub(r'\d+', '', cleaned_name))

    if '_' in cleaned_name:
        variations.add(cleaned_name.split('_')[0])
    if '.' in cleaned_name:
        variations.add(cleaned_name.split('.')[0])

    variations.add(cleaned_name.replace('_', ''))
    variations.add(re.sub(r'[^a-zA-Z]', '', cleaned_name))

    return [v for v in variations if v and len(v) > 1]

def align_metrics_with_code(df_metrics, code_files_dict):
    """Aligne explicitement les métriques avec le code source correspondant"""
    aligned_metrics = []
    aligned_code = []
    aligned_labels = []
    aligned_filenames = []

    matches_count = 0
    partial_matches_count = 0
    no_matches_count = 0

    # Créer un index des fichiers code pour recherche rapide
    code_index = {}
    for code_filename, code_content in code_files_dict.items():
        code_variations = extract_filename_from_path(code_filename)
        for variation in code_variations:
            if variation:
                code_index[variation] = (code_content, code_filename)

    print(f"✓ Index créé avec {len(code_index)} variations de noms de code")

    for idx, row in df_metrics.iterrows():
        # Extraire le nom de fichier des métriques
        if 'name' in row:
            metric_variations = extract_filename_from_path(str(row['name']))
        else:
            metric_variations = [f"file{idx}"]

        # Chercher le code correspondant
        matching_code = None
        match_type = "none"
        matched_filename = ""

        # Recherche exacte
        for metric_var in metric_variations:
            if metric_var in code_index:
                matching_code, matched_filename = code_index[metric_var]
                match_type = "exact"
                matches_count += 1
                break

        # Recherche partielle
        if matching_code is None:
            for metric_var in metric_variations:
                for code_var, (code_content, filename) in code_index.items():
                    if (metric_var and code_var and
                        (metric_var in code_var or code_var in metric_var)):
                        if len(metric_var) > 3 and len(code_var) > 3:
                            matching_code = code_content
                            matched_filename = filename
                            match_type = "partial"
                            partial_matches_count += 1
                            break
                if matching_code:
                    break

        # Fallback
        if matching_code is None and code_files_dict:
            matching_code, matched_filename = next(iter(code_index.values()))
            match_type = "fallback"
            no_matches_count += 1

        if matching_code is not None:
            # Préparer les métriques
            metric_data = {}
            for col, value in row.items():
                if col not in ['name', 'bug'] and pd.api.types.is_numeric_dtype(type(value)):
                    metric_data[col] = float(value) if pd.notna(value) else 0.0

            aligned_metrics.append(metric_data)
            aligned_code.append(matching_code)
            aligned_labels.append(1 if row.get('bug', 0) > 0 else 0)
            aligned_filenames.append(f"{matched_filename}_{match_type}")

    print(f"✓ Alignement: {matches_count} exacts, {partial_matches_count} partiels, {no_matches_count} fallbacks")
    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

def load_and_align_data(metrics_path, code_path):
    """Charge et aligne les métriques avec le code source"""
    # Charger les métriques
    df_metrics = pd.read_csv(metrics_path)
    print(f"✓ {len(df_metrics)} métriques chargées depuis {os.path.basename(metrics_path)}")

    # Charger tous les fichiers code
    code_files_dict = {}
    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        file_path = os.path.join(root, file)
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            code_content = f.read().strip()
                            if code_content and len(code_content) > 10:
                                code_files_dict[file] = code_content
                    except Exception as e:
                        continue

    print(f"✓ {len(code_files_dict)} fichiers code chargés depuis {os.path.basename(code_path)}")

    if not code_files_dict:
        print("⚠️ Aucun fichier code trouvé!")
        return [], [], [], []

    # Aligner les données
    aligned_metrics, aligned_code, aligned_labels, aligned_filenames = align_metrics_with_code(
        df_metrics, code_files_dict
    )

    print(f"✓ {len(aligned_metrics)} échantillons alignés sur {len(df_metrics)} métriques originales")

    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

# FONCTIONS DE PRÉTRAITEMENT
def preprocess_aligned_metrics(metrics_list, labels):
    """Prétraite les métriques alignées"""
    if not metrics_list:
        return np.array([]), np.array([]), None

    df_metrics = pd.DataFrame(metrics_list)

    # Gérer les valeurs manquantes
    if df_metrics.isnull().any().any():
        df_metrics = df_metrics.fillna(df_metrics.median())

    # Supprimer les features à variance nulle
    var_threshold = 0.01
    variances = df_metrics.var()
    low_var_cols = variances < var_threshold
    if low_var_cols.any():
        df_metrics = df_metrics.loc[:, ~low_var_cols]
        print(f"✓ {low_var_cols.sum()} colonnes à faible variance supprimées")

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(df_metrics)

    print(f"✓ Métriques prétraitées: {features_scaled.shape}")
    return features_scaled, np.array(labels), scaler

def apply_smote_with_realignment(X, y, code_tokens, filenames):
    """Applique SMOTE et réaligne correctement les tokens de code ET les filenames"""
    print(f"Distribution avant SMOTE: {Counter(y)}")

    original_size = len(X)

    # Appliquer SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)

    new_size = len(X_resampled)
    print(f"Distribution après SMOTE: {Counter(y_resampled)}")

    # Réaligner les tokens de code ET les filenames
    if new_size <= original_size:
        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][:new_size],
            'attention_mask': code_tokens['attention_mask'][:new_size]
        }
        filenames_resampled = filenames[:new_size]
    else:
        indices = list(range(original_size)) + \
                 list(np.random.randint(0, original_size, new_size - original_size))

        code_tokens_resampled = {
            'input_ids': code_tokens['input_ids'][indices],
            'attention_mask': code_tokens['attention_mask'][indices]
        }
        filenames_resampled = [filenames[i] for i in indices]

    print(f"✓ Tokens et filenames réalignés: {original_size} → {new_size} échantillons")
    return X_resampled, y_resampled, code_tokens_resampled, filenames_resampled

def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    print(f"✓ Poids des classes: {weights}")
    return weight_tensor

def prepare_aligned_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset avec alignement explicite"""
    # Charger et aligner les données
    metrics_list, code_lines, labels, filenames = load_and_align_data(metrics_path, code_path)

    if not metrics_list:
        raise ValueError("Aucune donnée alignée trouvée")

    # Prétraiter les métriques
    features, labels, scaler = preprocess_aligned_metrics(metrics_list, labels)

    # Tokeniser le code AVANT SMOTE
    original_code_tokens = tokenizer(
        code_lines,
        padding=True,
        truncation=True,
        return_tensors='pt',
        max_length=Config.MAX_CODE_LENGTH
    )

    # Équilibrer les classes SI demandé avec réalignement correct
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_tokens, filenames = apply_smote_with_realignment(
                features, labels, original_code_tokens, filenames
            )
            print("✓ SMOTE appliqué avec réalignement correct")
            return features, code_tokens, labels, scaler, filenames

        except Exception as e:
            print(f"Erreur équilibrage: {e}, utilisation des données originales")

    return features, original_code_tokens, labels, scaler, filenames

# FONCTIONS D'ÉVALUATION
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels, _ in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# FONCTION D'ENTRAÎNEMENT
def train_model(model_type="hybrid"):
    """Entraîne un modèle avec alignement explicite"""
    print(f"\n{'='*60}")
    print(f"ENTRAÎNEMENT: {model_type.upper()}")
    print(f"{'='*60}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Chemins des données
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/xerces/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/xerces/"

    train_versions = ["1.1", "1.2", "1.3"]
    val_version = "1.4.4"

    # Charger les données d'entraînement
    all_train_features = []
    all_train_labels = []
    all_train_filenames = []
    all_train_code_tokens = {'input_ids': [], 'attention_mask': []}

    for version in train_versions:
        metrics_path = f"{base_metrics_path}xerces-{version}.csv"
        code_path = f"{base_code_path}xerces-{version}"

        try:
            features, code_tokens, labels, _, filenames = prepare_aligned_dataset(
                metrics_path, code_path, tokenizer, apply_balancing=True
            )

            all_train_features.append(features)
            all_train_labels.extend(labels)
            all_train_filenames.extend(filenames)

            all_train_code_tokens['input_ids'].append(code_tokens['input_ids'])
            all_train_code_tokens['attention_mask'].append(code_tokens['attention_mask'])

            print(f"✓ Version {version}: {len(features)} échantillons alignés")

        except Exception as e:
            print(f"✗ Erreur avec la version {version}: {e}")
            continue

    if not all_train_features:
        raise ValueError("Aucune donnée d'entraînement chargée")

    # Concaténer toutes les données d'entraînement
    X_train = np.vstack(all_train_features)
    y_train = np.array(all_train_labels)

    # Concaténer les tokens
    all_train_code = {
        'input_ids': torch.cat(all_train_code_tokens['input_ids']),
        'attention_mask': torch.cat(all_train_code_tokens['attention_mask'])
    }

    print(f"✓ Données d'entraînement finales: {X_train.shape[0]} échantillons")
    print(f"✓ Distribution: {Counter(y_train)}")
    print(f"✓ Dimension des features: {X_train.shape[1]}")

    # Ajustement automatique si mismatch
    min_length = min(len(X_train), len(y_train), len(all_train_filenames), all_train_code['input_ids'].shape[0])
    if min_length < len(X_train):
        X_train = X_train[:min_length]
        y_train = y_train[:min_length]
        all_train_filenames = all_train_filenames[:min_length]
        all_train_code = {
            'input_ids': all_train_code['input_ids'][:min_length],
            'attention_mask': all_train_code['attention_mask'][:min_length]
        }
        print(f"✓ Données ajustées à {min_length} échantillons")

    # Charger les données de validation
    print("\nChargement des données de validation...")
    metrics_val_path = f"{base_metrics_path}xerces-{val_version}.csv"
    code_val_path = f"{base_code_path}xerces-{val_version}"

    X_val, code_val_tokens, y_val, _, val_filenames = prepare_aligned_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    print(f"✓ Données de validation: {X_val.shape[0]} échantillons")

    # Ajustement automatique de la validation
    min_length_val = min(len(X_val), len(y_val), len(val_filenames), code_val_tokens['input_ids'].shape[0])
    if min_length_val < len(X_val):
        X_val = X_val[:min_length_val]
        y_val = y_val[:min_length_val]
        val_filenames = val_filenames[:min_length_val]
        code_val_tokens = {
            'input_ids': code_val_tokens['input_ids'][:min_length_val],
            'attention_mask': code_val_tokens['attention_mask'][:min_length_val]
        }
        print(f"✓ Validation ajustée à {min_length_val} échantillons")

    # Vérifier la cohérence des features
    assert X_train.shape[1] == X_val.shape[1], f"Mismatch features: train{X_train.shape[1]} vs val{X_val.shape[1]}"

    # Créer les datasets
    train_ds = AlignedBugDataset(X_train, all_train_code, y_train, all_train_filenames)
    val_ds = AlignedBugDataset(X_val, code_val_tokens, y_val, val_filenames)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Initialiser le modèle selon la stratégie
    if model_type == "full_pretrained":
        model = FullPretrainedModel().to(Config.DEVICE)
        model_name = "Full Pretrained"
    else:
        model = HybridTransBugModel().to(Config.DEVICE)
        model_name = "Hybrid"

    print(f"✓ Modèle {model_name} initialisé avec LazyLinear")

    # Initialiser les LazyLinear avec une passe forward
    print("🔧 Initialisation des couches LazyLinear...")
    with torch.no_grad():
        # Passe forward d'initialisation
        sample_metrics = torch.randn(2, X_train.shape[1]).to(Config.DEVICE)
        sample_code_tokens = {
            'input_ids': torch.randint(0, 1000, (2, 10)).to(Config.DEVICE),
            'attention_mask': torch.ones(2, 10).to(Config.DEVICE)
        }
        _ = model(sample_metrics, sample_code_tokens)
    print("✓ LazyLinear initialisés")

    # Optimiseur avec learning rates différenciés
    if model_type == "full_pretrained":
        # Pour le modèle entièrement pré-entraîné, utiliser des LR plus faibles
        optimizer = torch.optim.AdamW([
            {'params': model.metrics_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters()
                       if 'metrics_encoder' not in n and 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)
    else:
        # Pour le modèle hybride, seul le decoder est pré-entraîné
        optimizer = torch.optim.AdamW([
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    # Loss avec pondération
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0
    best_model_state = None

    # ENTRAÎNEMENT
    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels, _) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds) if train_preds else 1
        train_acc = accuracy_score(train_labels_list, train_preds) if train_labels_list else 0
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss if val_loss else 0)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
            print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
            print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 10:
            print(f"Early stopping déclenché à l'epoch {epoch+1}")
            break

    # Charger le meilleur modèle
    if best_model_state:
        model.load_state_dict(best_model_state)

    print(f"\n{model_name.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'model_name': model_name,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

# FONCTION PRINCIPALE DE COMPARAISON
def main():
    """Fonction principale pour comparer les deux stratégies"""
    print("🚀 COMPARAISON DES STRATÉGIES TRANSFORMERS POUR LA DÉTECTION DE BUGS")
    print("=" * 80)
    print("STRATÉGIE 1: HYBRIDE - Encoder from scratch + Decoder pré-entraîné")
    print("              • Métriques → Encoder transformer from scratch")
    print("              • Code source → Decoder RoBERTa pré-entraîné")
    print("              • Attention croisée entre les deux représentations")
    print()
    print("STRATÉGIE 2: FULL PRETRAINED - Encoder et Decoder pré-entraînés")
    print("              • Métriques → Encoder RoBERTa pré-entraîné")
    print("              • Code source → Decoder RoBERTa pré-entraîné")
    print("              • Attention croisée entre les deux représentations")
    print("=" * 80)
    print("AVANTAGE LazyLinear: Adaptation automatique aux dimensions des features")
    print("=" * 80)

    results = {}

    try:
        # Entraîner le modèle hybride
        print("\n" + "="*50)
        print("1. STRATÉGIE HYBRIDE (Encoder from scratch + Decoder pré-entraîné)")
        print("="*50)
        model_hybrid, results_hybrid = train_model("hybrid")
        results['hybrid'] = results_hybrid

        # Entraîner le modèle entièrement pré-entraîné
        print("\n" + "="*50)
        print("2. STRATÉGIE FULL PRETRAINED (Encoder et Decoder pré-entraînés)")
        print("="*50)
        model_full, results_full = train_model("full_pretrained")
        results['full_pretrained'] = results_full

        # COMPARAISON DÉTAILLÉE
        print("\n" + "="*80)
        print("COMPARAISON FINALE DES STRATÉGIES")
        print("="*80)

        print(f"{'Métrique':<20} {'Hybride':<12} {'Full Pretrained':<16} {'Différence':<12} {'Meilleur':<10}")
        print("-" * 80)

        metrics = [
            ('F1-Score', results_hybrid['best_f1'], results_full['best_f1']),
            ('G-Mean', results_hybrid['final_gmean'], results_full['final_gmean']),
            ('Accuracy', results_hybrid['final_accuracy'], results_full['final_accuracy']),
            ('Sensitivity', results_hybrid['final_sensitivity'], results_full['final_sensitivity']),
            ('Specificity', results_hybrid['final_specificity'], results_full['final_specificity'])
        ]

        for metric_name, hybrid_score, full_score in metrics:
            difference = hybrid_score - full_score
            better = "Hybride" if hybrid_score > full_score else "Full Pretrained" if full_score > hybrid_score else "Égalité"
            diff_sign = "+" if difference > 0 else ""
            print(f"{metric_name:<20} {hybrid_score:<12.4f} {full_score:<16.4f} {diff_sign}{difference:<11.4f} {better:<10}")

        # ANALYSE DES RÉSULTATS
        print("\n" + "="*80)
        print("ANALYSE DES RÉSULTATS")
        print("="*80)

        if results_hybrid['best_f1'] > results_full['best_f1']:
            best_model = "HYBRIDE"
            best_score = results_hybrid['best_f1']
            advantage = f"+{(results_hybrid['best_f1'] - results_full['best_f1']):.4f}"
            print(f"🎯 STRATÉGIE GAGNANTE: {best_model}")
            print(f"📊 Avantage F1-Score: {advantage}")
            print("💡 INTERPRÉTATION: L'encoder from scratch pour les métriques combiné")
            print("   avec le decoder pré-entraîné pour le code source donne de meilleurs résultats.")
            print("   La flexibilité des LazyLinear permet une meilleure adaptation aux métriques.")

        else:
            best_model = "FULL PRETRAINED"
            best_score = results_full['best_f1']
            advantage = f"+{(results_full['best_f1'] - results_hybrid['best_f1']):.4f}"
            print(f"🎯 STRATÉGIE GAGNANTE: {best_model}")
            print(f"📊 Avantage F1-Score: {advantage}")
            print("💡 INTERPRÉTATION: La connaissance pré-entraînée pour les deux modalités")
            print("   (métriques et code) combinée avec LazyLinear donne de meilleurs résultats.")

        print(f"\n✅ Résultats sauvegardés dans: {Config.SAVE_PATH}")

    except Exception as e:
        print(f"❌ Erreur lors de la comparaison: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()